In [458]:
# Importe

import re
import pandas as pd
import numpy as np


### RIS

In [459]:
# RIS lesen

RIS_PATH = "../../data/metadata/raw/gbv-ris.ris"

def read_ris_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()
    return content

ris_file = read_ris_file(RIS_PATH)

In [460]:
# RIS zu DataFrame

def ris_to_df(ris):
    rows = []
    eintrag = {}
    key_counts = {}
    
    for line in ris.splitlines():
        line = line.strip()
        if line == "ER  -":
            rows.append(eintrag)
            eintrag = {}
            key_counts = {}
        elif "  - " in line:
            key, value = line.split("  - ", 1)
            key = key.strip()
            value = value.strip()
            if key in eintrag:
                i = 2
                while f"{key}_{i}" in eintrag:
                    i += 1
                eintrag[f"{key}_{i}"] = value
            else:
                eintrag[key] = value
    
    return pd.DataFrame(rows)

ris_df = ris_to_df(ris_file)


# RIS-Datenbereinigung

mask = ris_df['A2'].notna()
ris_df.loc[mask, 'A1'] = ris_df.loc[mask, 'A2']
ris_df = ris_df.drop(columns=["PB", 
                              "CY",
                              "A2", 
                              "U1", 
                              "TY", 
                              "H2", "H2_2", 
                              "S1", "S2", 
                              "L3",  
                              "KW", "KW_2",
                              "T2", "T2_2", "T2_3", "T2_4", "T2_5", "T2_6", "T2_7", "T3", 
                              "N1", "N1_2", "N1_3"])

# PY (Erscheinungsjahr) bereinigen: einzelne Ausreisser (z.B. 2001 = Digitalisierungsjahr
# des DFG-Projekts, faelschlich als PY erfasst -- betrifft nur 11 von ~5986 Zeilen)
# ausserhalb des plausiblen Zeitraums der Schriftenreihen (1700-1900) verwerfen
py_numeric = pd.to_numeric(ris_df['PY'], errors='coerce')
ris_df.loc[(py_numeric < 1700) | (py_numeric > 1900), 'PY'] = None

def normalize_caps(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return value
    
    def fix_word(word):
        if word.isupper():
            return word.capitalize()
        return word

    return ' '.join(fix_word(w) for w in str(value).split(' '))

ris_df['A1'] = ris_df['A1'].apply(normalize_caps)

In [461]:
# Vollanzeige lesen und in Liste splitten

VA_PATH = "../../data/metadata/raw/gbv-vollanzeige.txt"

def read_va_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()
    return content

va_file = read_va_file(VA_PATH)
va_file = re.split(r'(?=Aufsatz / Teil:)', va_file.strip())
va_file.pop(0) 

''

### VA

In [462]:
# VA zu Dataframe

labels = [
    'Aufsatz / Teil:',
    'VerfasserIn:',
    'Veröffentlichungsangabe:',
    'Umfang:',
    'Anmerkung:',
    'Bibliogr. Zusammenhang:',
    'Gesamttitel:',
    'Volltext:',
    'Lokale Schlagwörter:',
    'Signatur:',
    'Schlagwörter:',
    'Standort:',
    'Schriftenreihe:'   
]

pattern = '(' + '|'.join(re.escape(label) for label in labels) + ')'

va_liste_neu = []
for eintrag in va_file:
    eintrag = re.sub(r'\s+', ' ', eintrag).strip()
    teile = re.split(pattern, eintrag)
    
    row = {}
    for i in range(1, len(teile), 2):
        key = teile[i].strip().rstrip(':')
        value = teile[i+1].strip() if i+1 < len(teile) else ''
        if not value:
            continue
        if key in row:
            i_count = 2
            while f"{key}_{i_count}" in row:
                i_count += 1
            row[f"{key}_{i_count}"] = value
        else:
            row[key] = value
    
    va_liste_neu.append(row)

va_df = pd.DataFrame(va_liste_neu)

In [463]:
va_df

,Aufsatz / Teil,VerfasserIn,Veröffentlichungsangabe,Veröffentlichungsangabe_2,Gesamttitel,Volltext,Signatur,Anmerkung,Schlagwörter,Bibliogr. Zusammenhang,...,Bibliogr. Zusammenhang_2,Schlagwörter_2,Gesamttitel_3,Volltext_3,Standort_2,Signatur_3,Anmerkung_2,VerfasserIn_2,Schriftenreihe,Anmerkung_3
0,Die elektromagnetische Drehung der Polarisatio...,"Kundt, August *1839-1894*",2001 (Original: 1884),Berlin : Berlin-Brandenburgische Akademie der ...,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 350,Vgl.: Zweite Mittheilung unter dem Titel: Über...,Akademiemitglied_v,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Über die Mittheilung des Tones longitudinal sc...,"KUNDT, August",2001 (Original: 1865),Berlin : Berlin-Brandenburgische Akademie der ...,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,NaN,Akademiemitglied_v,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Über Erzeugung stehender Schwingungen und Klan...,"KUNDT, August",2001 (Original: 1868),Berlin : Berlin-Brandenburgische Akademie der ...,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,NaN,Akademiemitglied_v,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Epigraphische Reiseberichte aus Spanien und Po...,"Hübner, Emil *1834-1901*",2001,Berlin : Berlin-Brandenburgische Akademie der ...,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,Vgl. 1860. MB S. 231-241; 324-332; 421-450; 59...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Über einige Bestandtheile der peripheren markh...,"Joseph, Max *1860-1932*",2001 (Original: 1888),Berlin : Berlin-Brandenburgische Akademie der ...,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 350,NaN,NaN,In: Sitzungsberichte der Königlich Preußischen...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8298,Über Potenzreihen,"KRONECKER, Leopold",Berlin : Berlin-Brandenburgische Akademie der ...,NaN,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,Monatsberichte der Königlich Preußischen Akade...,Akademiemitglied_v,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8299,Über die Ossification der Geweihe,"LIEBERKÜHN, Nathanael",Berlin : Berlin-Brandenburgische Akademie der ...,NaN,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,Monatsberichte der Königlich Preußischen Akade...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8300,Über eine Untersuchung des sogenannten capitol...,"JORDAN, Heinrich",Berlin : Berlin-Brandenburgische Akademie der ...,NaN,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,Monatsberichte der Königlich Preußischen Akade...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8301,"Über eine neue, mit Halieutaea verwandte Fisch...","PETERS, Wilhelm",Berlin : Berlin-Brandenburgische Akademie der ...,NaN,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,Monatsberichte der Königlich Preußischen Akade...,Akademiemitglied_v,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,mit 1 Tafel,NaN,NaN,NaN


In [464]:
# VA-Daten löschen

# Spalten löschen
mask = va_df['Anmerkung'].isna() & va_df['Umfang'].notna()
va_df.loc[mask, 'Anmerkung'] = va_df.loc[mask, 'Umfang']

va_df = va_df.drop(columns=["Volltext_2", "Volltext_3", 
                            "Schriftenreihe",
                            "Aufsatz / Teil", 
                            "Signatur", "Signatur_2", "Signatur_3", 
                            "Standort", "Standort_2", 
                            "Gesamttitel", "Gesamttitel_2", "Gesamttitel_3", 
                            "Schlagwörter", "Schlagwörter_2", "Lokale Schlagwörter",
                            "Veröffentlichungsangabe", "Veröffentlichungsangabe_2", "Veröffentlichungsangabe_3", "Veröffentlichungsangabe_4",
                            "VerfasserIn", "VerfasserIn_2",
                            "Bibliogr. Zusammenhang_2",
                            "Anmerkung_3",
                            "Umfang", "Umfang_2"
                            ])



In [465]:
va_df

,Volltext,Anmerkung,Bibliogr. Zusammenhang,Anmerkung_2
0,<a href='https://digilib.bbaw.de/digitallibrar...,Vgl.: Zweite Mittheilung unter dem Titel: Über...,NaN,NaN
1,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,NaN
2,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,NaN
3,<a href='https://digilib.bbaw.de/digitallibrar...,Vgl. 1860. MB S. 231-241; 324-332; 421-450; 59...,NaN,NaN
4,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,In: Sitzungsberichte der Königlich Preußischen...,NaN
...,...,...,...,...
8298,<a href='https://digilib.bbaw.de/digitallibrar...,Monatsberichte der Königlich Preußischen Akade...,NaN,NaN
8299,<a href='https://digilib.bbaw.de/digitallibrar...,Monatsberichte der Königlich Preußischen Akade...,NaN,NaN
8300,<a href='https://digilib.bbaw.de/digitallibrar...,Monatsberichte der Königlich Preußischen Akade...,NaN,NaN
8301,<a href='https://digilib.bbaw.de/digitallibrar...,Monatsberichte der Königlich Preußischen Akade...,NaN,mit 1 Tafel


In [466]:
# Anmerkungen und Bibliographischen Zusammenhang mergen

# Bibliographischen Zusammenhang aufräumen
suchstrings_bib_zus = "Erscheint auch als|Sonderdruck|Druckausg|Digitalisierte Akademieschriften"
maske1 = va_df["Bibliogr. Zusammenhang"].str.contains(suchstrings_bib_zus, na=False)
leer1 = maske1 & va_df["Anmerkung_2"].isna()
gefuellt1 = maske1 & va_df["Anmerkung_2"].notna()
va_df.loc[leer1, "Anmerkung_2"] = va_df.loc[leer1, "Anmerkung"]
va_df.loc[gefuellt1, "Anmerkung_2"] = va_df.loc[gefuellt1, "Anmerkung_2"] + "; " + va_df.loc[gefuellt1, "Anmerkung"]
va_df.loc[maske1, "Bibliogr. Zusammenhang"] = np.nan
va_df["Bibliogr. Zusammenhang"] = va_df["Bibliogr. Zusammenhang"].str.split(";").str[-1]
va_df.loc[va_df["Bibliogr. Zusammenhang"].str.contains("In:", na=False), "Bibliogr. Zusammenhang"] = np.nan

# Anmerkungen aufräumen
suchstrings_anmerkung = "Tafel|Tabelle|Blätter|Rezension|Réponse|Neue Versuche|Nachträge|Nachtrag|Vgl.|Vergl.|Fortsetzung|Auszug|Antwort|Zweiter Theil|Zweiter Bericht|Zweite Mittheilung"
maske2 = va_df["Anmerkung"].str.contains(suchstrings_anmerkung, na=False)
leer2 = maske2 & va_df["Anmerkung_2"].isna()
gefuellt2 = maske2 & va_df["Anmerkung_2"].notna()
va_df.loc[leer2, "Anmerkung_2"] = va_df.loc[leer2, "Anmerkung"]
va_df.loc[gefuellt2, "Anmerkung_2"] = va_df.loc[gefuellt2, "Anmerkung_2"] + "; " + va_df.loc[gefuellt2, "Anmerkung"]
va_df.loc[maske2, "Anmerkung"] = np.nan
va_df["Anmerkung"] = va_df["Anmerkung"].str.split("Übers.").str[0]
va_df["Anmerkung"] = va_df["Anmerkung"].str.split("Veröffentlichungsangabe").str[0]
va_df["Anmerkung"] = va_df["Anmerkung"].str.split(";").str[-1]

# Bibliographischer Zusammenhang und Anmerkung mergen
maske = va_df["Bibliogr. Zusammenhang"].notna()
va_df.loc[maske, "Anmerkung"] = va_df.loc[maske, "Bibliogr. Zusammenhang"]
va_df = va_df.drop(columns=["Bibliogr. Zusammenhang"])
va_df["Seiten"] = va_df["Anmerkung"]
va_df = va_df.drop(columns=["Anmerkung"])
va_df["Anmerkung"] = va_df["Anmerkung_2"]
va_df = va_df.drop(columns=["Anmerkung_2"])
va_df["Anmerkung"] = va_df["Anmerkung"].str.strip("\"'„“‚’")

In [467]:
# Seiten bearbeiten

# Fortsetzung aus Seiten extrahieren
maske = va_df["Seiten"].str.contains(" und ", na=False)
va_df.loc[maske, "Fortsetzung"] = "ebenda. " + va_df.loc[maske, "Seiten"].str.split(" und ").str[1]
va_df.loc[maske, "Seiten"] = va_df.loc[maske, "Seiten"].str.split(" und ").str[0]

# Klassen aus Seiten extrahieren (und droppen)
va_df["Klasse"] = va_df["Seiten"].str.extract(r"^([^,]*,)")
va_df["Seiten"] = va_df["Seiten"].str.replace(r"^[^,]*,", "", regex=True)
va_df["Seiten"] = va_df["Seiten"].str.replace(r"\s+", "", regex=True)
va_df["Klasse"] = va_df["Klasse"].str.strip().str.rstrip(",")
va_df = va_df.drop(columns=["Klasse"])

# SP und EP extrahieren
def berechne_sp_ep(df, spalte, sp_name, ep_name):
    # Klammern entfernen
    df[spalte] = df[spalte].str.replace(r"[()]", "", regex=True)

    # Maske: Zeilen, in denen die Spalte mit "S" beginnt
    maske_s = df[spalte].str.startswith("S", na=False)
    werte_s = df.loc[maske_s, spalte].str.findall(r"\d+|[IVXLCDM]+"
)

    # Fall 1: nur ein Wert -> SP = EP
    einzelwert = werte_s[werte_s.str.len() == 1]
    df.loc[einzelwert.index, sp_name] = einzelwert.str[0]
    df.loc[einzelwert.index, ep_name] = einzelwert.str[0]

    # Fall 2: zwei Werte -> SP = erster, EP = zweiter
    zweiwerte = werte_s[werte_s.str.len() >= 2]
    df.loc[zweiwerte.index, sp_name] = zweiwerte.str[0]
    df.loc[zweiwerte.index, ep_name] = zweiwerte.str[1]

    # Maske: Zeilen, die NICHT mit "S" beginnen
    maske_kein_s = ~df[spalte].str.startswith("S", na=False)

    # Fall 3: Werte mit "-" getrennt -> SP = erster, EP = zweiter
    maske_bindestrich = maske_kein_s & df[spalte].str.contains("-", na=False)
    werte_bindestrich = df.loc[maske_bindestrich, spalte].str.findall(r"\d+|[IVXLCDM]+"
)
    df.loc[werte_bindestrich.index, sp_name] = werte_bindestrich.str[0]
    df.loc[werte_bindestrich.index, ep_name] = werte_bindestrich.str[1]

    # Fall 4: "S" kommt nach dem Wert -> SP = 1, EP = Wert
    maske_s_danach = maske_kein_s & ~df[spalte].str.contains("-", na=False) & df[spalte].str.contains("S", na=False)
    werte_s_danach = df.loc[maske_s_danach, spalte].str.findall(r"\d+|[IVXLCDM]+"
)
    df.loc[werte_s_danach.index, sp_name] = "1"
    df.loc[werte_s_danach.index, ep_name] = werte_s_danach.str[0]

    # Fall 5: kein "-", kein "S", ein Wert -> SP = 1, EP = Wert
    maske_nur_wert = maske_kein_s & ~df[spalte].str.contains("-", na=False) & ~df[spalte].str.contains("S", na=False)
    werte_nur_wert = df.loc[maske_nur_wert, spalte].str.findall(r"\d+|[IVXLCDM]+"
)
    df.loc[werte_nur_wert.index, sp_name] = "1"
    df.loc[werte_nur_wert.index, ep_name] = werte_nur_wert.str[0]

    # SP > EP -> erste Ziffer von SP entfernen (nur bei arabischen Zahlen wirksam,
    # da pd.to_numeric römische Ziffern zu NaN macht und der Vergleich dann False ergibt)
    sp_num = pd.to_numeric(df[sp_name], errors="coerce")
    ep_num = pd.to_numeric(df[ep_name], errors="coerce")
    maske_sp_groesser = sp_num > ep_num
    df.loc[maske_sp_groesser, sp_name] = df.loc[maske_sp_groesser, sp_name].str[1:]

    # Fall: Spalte leer/NaN -> SP, EP explizit leeren (überschreibt alles Vorherige)
    maske_leer = df[spalte].isna() | (df[spalte].str.strip() == "")
    df.loc[maske_leer, sp_name] = None
    df.loc[maske_leer, ep_name] = None

# Aufruf für beide Spalten
berechne_sp_ep(va_df, "Seiten", "SP", "EP")
va_df = va_df.drop(columns=["Seiten"])

va_df 

,Volltext,Anmerkung,Fortsetzung,SP,EP
0,<a href='https://digilib.bbaw.de/digitallibrar...,Vgl.: Zweite Mittheilung unter dem Titel: Über...,NaN,None,None
1,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,None,None
2,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,None,None
3,<a href='https://digilib.bbaw.de/digitallibrar...,Vgl. 1860. MB S. 231-241; 324-332; 421-450; 59...,NaN,None,None
4,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,1321,1330
...,...,...,...,...,...
8298,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,53,58
8299,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,264,267
8300,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,526,548
8301,<a href='https://digilib.bbaw.de/digitallibrar...,mit 1 Tafel,NaN,736,742


In [468]:
# Informationen aus Anmerkungen extrahieren

# Tafel-Angaben extrahieren
maske_mit = va_df["Anmerkung"].str.match(r"^(mit|und)", na=False)
pattern_mit_trenner = r"^((?:mit|und).*?)(?:(?<!\w)-(?!\w)|(?<!Fig)(?<!\d)\.)"
extrahiert = va_df.loc[maske_mit, "Anmerkung"].str.extract(pattern_mit_trenner)[0]
maske_kein_trenner = extrahiert.isna()
extrahiert.loc[maske_kein_trenner] = va_df.loc[maske_mit, "Anmerkung"][maske_kein_trenner]
va_df.loc[maske_mit, "Abbilder"] = extrahiert
va_df.loc[maske_mit, "Anmerkung"] = va_df.loc[maske_mit, "Anmerkung"].str.replace(pattern_mit_trenner, "", regex=True, n=1)
va_df.loc[maske_mit & maske_kein_trenner.reindex(va_df.index, fill_value=False), "Anmerkung"] = ""

# Übersetzungsinformationen extrahieren (bleibt erhalten, wird spaeter in Textbeziehungen aufgeloest)
pattern_ueb = r"(Übersetzung.*|Deutsch unter dem Titel.*|In französischer Übersetzung.*|Original unter dem Titel.*)$"
maske_ueb = va_df["Anmerkung"].str.contains(
    r"Übersetzung|Deutsch unter dem Titel|In französischer Übersetzung|Original unter dem Titel",
    na=False
)
va_df.loc[maske_ueb, "Übersetzung/Original"] = va_df.loc[maske_ueb, "Anmerkung"].str.extract(pattern_ueb)[0]
va_df.loc[maske_ueb, "Anmerkung"] = va_df.loc[maske_ueb, "Anmerkung"].str.replace(pattern_ueb, "", regex=True)

# Anworten extrahieren
pattern_ant = r"(Antwort von.*|Autwort von.*|Résponse de.*)$"
maske_ant = va_df["Anmerkung"].str.contains(
    r"Antwort von.*|Autwort von.*|Résponse de.*",
    na=False
)
va_df.loc[maske_ant, "Antwort"] = va_df.loc[maske_ant, "Anmerkung"].str.extract(pattern_ant)[0]
va_df.loc[maske_ant, "Anmerkung"] = va_df.loc[maske_ant, "Anmerkung"].str.replace(pattern_ant, "", regex=True)




In [469]:
# Reinigungen mit Informationsverlust
va_df["Anmerkung"] = va_df["Anmerkung"].str.replace(r"[\[\]()]", "", regex=True)
va_df["Anmerkung"] = va_df["Anmerkung"].str.strip()
va_df["Anmerkung"] = va_df["Anmerkung"].str.replace(r"^-|-$", "", regex=True)
va_df["Anmerkung"] = va_df["Anmerkung"].str.strip()
va_df["Anmerkung"] = va_df["Anmerkung"].replace("", np.nan)

va_df["Abbilder"] = va_df["Abbilder"].str.replace(r"(\[|;|Vgl|Second|die|und SB|\().*$", "", regex=True)
va_df["Abbilder"] = va_df["Abbilder"].str.strip().str.rstrip(".")

maske_loeschen = va_df["Anmerkung"].str.match(r"^(\d|In franzosisch|Abhandlungen|Abh.|=|Bezeichnet als|Bericht|Einige|Druckfehle|Monatsberichte|Uranios|S.|SB|Auch in Formey|Addition|Anhang zu dieser Abhandlung)", na=False, case=False)
va_df.loc[maske_loeschen, "Anmerkung"] = np.nan

### MERGE

In [470]:
# RIS und VA zusammenführen

# Link für Merge herstellen
def extract_link(text):
    if pd.isna(text):
        return text
    match = re.search(r"href=['\"](.*?)['\"]", text)
    if match:
        link = match.group(1)
        link = re.sub(r'\s+', '', link)  # entfernt alle Leerzeichen/Umbrüche im Link
        return link
    return None

va_df['Volltext'] = va_df['Volltext'].apply(extract_link)
merged_df = pd.merge(
    va_df, ris_df,
    left_on="Volltext", right_on="UR",
    how="outer", indicator=True
)

# Reinigung
merged_df = merged_df.drop(columns=["Volltext", "_merge"])
merged_df = merged_df.drop_duplicates(subset=["UR"], keep="first")
merged_df = merged_df[merged_df['ID'] != "53886186X"]

# Schriftenreihe und Band aus der UR extrahieren
pattern = r'Bibliothek\.tiff/([^/]+)/([^/]+)/tif/'
extracted = merged_df['UR'].str.extract(pattern)
merged_df['SR'] = extracted[0]
merged_df['BD'] = extracted[1]

In [471]:
# Seiten manuell nachtragen
#
# Alle hier gesetzten Seiten werden in "Seiten_Quelle" als "manuell" markiert
# (siehe Zelle "Fehlende Seitenzahlen ... schaetzen").

seiten_vorher = merged_df[["SP", "EP"]].copy()

merged_df.loc[merged_df["ID"] == "094932654", "SP"] = "1"
merged_df.loc[merged_df["ID"] == "094945292", "SP"] = "1"
merged_df.loc[merged_df["ID"] == "094945292", "EP"] = "176"
merged_df.loc[merged_df["ID"] == "094958130", "SP"] = "111"
merged_df.loc[merged_df["ID"] == "09496775X", "SP"] = "394"
merged_df.loc[merged_df["ID"] == "09496775X", "EP"] = "416"
merged_df.loc[merged_df["ID"] == "094917906", "SP"] = "187"
merged_df.loc[merged_df["ID"] == "094917906", "EP"] = "187"
# 06-samml 1803: Antrittsreden von Eytelwein und Spalding -- im Druck ohne
# Seitenzahlen; 0-0 ist gewollt (laut Pruefung im Viewer).
merged_df.loc[merged_df["ID"] == "094955220", "SP"] = "0"
merged_df.loc[merged_df["ID"] == "094955220", "EP"] = "0"
merged_df.loc[merged_df["ID"] == "094969892", "SP"] = "0"
merged_df.loc[merged_df["ID"] == "094969892", "EP"] = "0"
merged_df.loc[merged_df["ID"] == "094902305", "SP"] = "329"
merged_df.loc[merged_df["ID"] == "094902305", "EP"] = "359"
merged_df.loc[merged_df["ID"] == "094926573", "SP"] = "677"
merged_df.loc[merged_df["ID"] == "094926573", "EP"] = "706"
merged_df.loc[merged_df["ID"] == "094971730", "SP"] = "329"
merged_df.loc[merged_df["ID"] == "094971730", "EP"] = "362"
merged_df.loc[merged_df["ID"] == "094972605", "SP"] = "628"
merged_df.loc[merged_df["ID"] == "094972605", "EP"] = "636"
# 08-verh 1852: Mitscherlich, Ueber die Waerme, welche frei wird, wenn die
# Krystalle des Schwefels ... in die andere Form uebergehen -- Katalog "639-039";
# Seiten laut Pruefung im Viewer 636-639.
merged_df.loc[merged_df["ID"] == "094977550", "SP"] = "636"
merged_df.loc[merged_df["ID"] == "094977550", "EP"] = "639"
merged_df.loc[merged_df["ID"] == "09495688X", "SP"] = "248"
merged_df.loc[merged_df["ID"] == "09495688X", "EP"] = "252"
merged_df.loc[merged_df["ID"] == "094969213", "SP"] = "234"
merged_df.loc[merged_df["ID"] == "094969213", "EP"] = "255"
merged_df.loc[merged_df["ID"] == "094936366", "SP"] = "62"
merged_df.loc[merged_df["ID"] == "094936366", "EP"] = "70"
merged_df.loc[merged_df["ID"] == "094897212", "SP"] = "747"
merged_df.loc[merged_df["ID"] == "094897212", "EP"] = "772"
merged_df.loc[merged_df["ID"] == "094904707", "SP"] = "82"
merged_df.loc[merged_df["ID"] == "094904707", "EP"] = "119"
merged_df.loc[merged_df["ID"] == "094966303", "SP"] = "123"
merged_df.loc[merged_df["ID"] == "094966303", "EP"] = "130"
merged_df.loc[merged_df["ID"] == "094896968", "SP"] = "629"
merged_df.loc[merged_df["ID"] == "094896968", "EP"] = "640"
merged_df.loc[merged_df["ID"] == "094925887", "SP"] = "759"
merged_df.loc[merged_df["ID"] == "094925887", "EP"] = "763"
merged_df.loc[merged_df["ID"] == "094907757", "SP"] = "100"
merged_df.loc[merged_df["ID"] == "094907757", "EP"] = "106"
merged_df.loc[merged_df["ID"] == "094969604", "SP"] = "87"
merged_df.loc[merged_df["ID"] == "094969604", "EP"] = "89"
merged_df.loc[merged_df["ID"] == "094976732", "SP"] = "761"
merged_df.loc[merged_df["ID"] == "094976732", "EP"] = "782"
merged_df.loc[merged_df["ID"] == "094903123", "SP"] = "781"
merged_df.loc[merged_df["ID"] == "094903123", "EP"] = "786"
merged_df.loc[merged_df["ID"] == "094925895", "SP"] = "1035"
merged_df.loc[merged_df["ID"] == "094925895", "EP"] = "1042"
merged_df.loc[merged_df["ID"] == "09492483X", "SP"] = "21"
merged_df.loc[merged_df["ID"] == "09492483X", "EP"] = "32"
merged_df.loc[merged_df["ID"] == "094900248", "SP"] = "515"
merged_df.loc[merged_df["ID"] == "094900248", "EP"] = "526"
merged_df.loc[merged_df["ID"] == "094897174", "SP"] = "659"
merged_df.loc[merged_df["ID"] == "094897174", "EP"] = "664"
merged_df.loc[merged_df["ID"] == "094911495", "SP"] = "665"
merged_df.loc[merged_df["ID"] == "094911495", "EP"] = "667"
merged_df.loc[merged_df["ID"] == "094932069", "SP"] = "782"
merged_df.loc[merged_df["ID"] == "094932069", "EP"] = "815"
merged_df.loc[merged_df["ID"] == "094919364", "SP"] = "231"
merged_df.loc[merged_df["ID"] == "094919364", "EP"] = "241"
merged_df.loc[merged_df["ID"] == "094966222", "SP"] = "16"
merged_df.loc[merged_df["ID"] == "094966222", "EP"] = "113"

# 07-abh 1854: Katalog nennt fuer Ritter "33-303" (Versatz Bild/Seite 727 statt 427
# wie im ganzen Teil), richtig ist 333-363; die Burggrafschaft-Abhandlung (ohne
# Seiten im Katalog) folgt auf 364 und endet vor Ranke (S. 415).
merged_df.loc[merged_df["ID"] == "094904723", "SP"] = "333"
merged_df.loc[merged_df["ID"] == "094904723", "EP"] = "363"
merged_df.loc[merged_df["ID"] == "094933502", "SP"] = "364"
merged_df.loc[merged_df["ID"] == "094933502", "EP"] = "414"

# 07-abh 18991900 (eigene Seitenzaehlung je Abhandlung): Schaudinn,
# Generationswechsel von Trichosphaerium -- Katalog ohne Seitenangabe,
# laut Pruefung im Viewer S. 1-93.
merged_df.loc[merged_df["ID"] == "094956456", "SP"] = "1"
merged_df.loc[merged_df["ID"] == "094956456", "EP"] = "93"

# 08-verh 1840: Encke, Über die Störungen der Vesta ... : Auszug -- Katalog ohne
# Seitenangabe. Startseite 253 aus der Schaetzung ueber die Bildnummer,
# Endseite 259 laut Pruefung im Viewer.
merged_df.loc[merged_df["ID"] == "094974950", "SP"] = "253"
merged_df.loc[merged_df["ID"] == "094974950", "EP"] = "259"

# 08-verh 1841: Encke, Über die Masse des Merkur -- Katalog ohne Seitenangabe.
# Startseite 421 aus der Schaetzung ueber die Bildnummer, Endseite 423 laut
# Pruefung im Viewer.
merged_df.loc[merged_df["ID"] == "094950946", "SP"] = "421"
merged_df.loc[merged_df["ID"] == "094950946", "EP"] = "423"

# 08-verh 1845: Encke, Entdeckung und Elemente der Astraea -- Katalog ohne
# Seitenangabe. Startseite 405 aus der Schaetzung ueber die Bildnummer,
# Endseite 407 laut Pruefung im Viewer.
merged_df.loc[merged_df["ID"] == "094958246", "SP"] = "405"
merged_df.loc[merged_df["ID"] == "094958246", "EP"] = "407"

# Encke, Johann Franz -- Katalog ohne Seitenangaben; Seiten laut Pruefung im
# Viewer. Gedaechtnisreden/Berichte am Bandanfang stehen im roemisch
# gezaehlten Vorspann.
# 07-abh 1826: Gedaechtnissrede auf Johann Georg Tralles
merged_df.loc[merged_df["ID"] == "094952574", "SP"] = "XII"
merged_df.loc[merged_df["ID"] == "094952574", "EP"] = "XVII"
# 07-abh 1827: Gedaechtnissrede auf Johann Elert Bode
merged_df.loc[merged_df["ID"] == "094967563", "SP"] = "XI"
merged_df.loc[merged_df["ID"] == "094967563", "EP"] = "XXI"
# 07-abh 1828: Bericht ueber das Unternehmen der akademischen Sternkarten
merged_df.loc[merged_df["ID"] == "09496484X", "SP"] = "IX"
merged_df.loc[merged_df["ID"] == "09496484X", "EP"] = "XV"
# 07-abh 1844: Ueber den Cometen von Pons : Fuenfte Abhandlung
merged_df.loc[merged_df["ID"] == "094919291", "SP"] = "74"
merged_df.loc[merged_df["ID"] == "094919291", "EP"] = "99"
# 07-abh 1845: Ueber die Polhoehe der neuen Berliner Sternwarte
merged_df.loc[merged_df["ID"] == "094953171", "SP"] = "86"
merged_df.loc[merged_df["ID"] == "094953171", "EP"] = "114"
# 07-abh 1846: Gedaechtnissrede auf Bessel
merged_df.loc[merged_df["ID"] == "094925283", "SP"] = "XXI"
merged_df.loc[merged_df["ID"] == "094925283", "EP"] = "XLII"
# 07-abh 1849: Gedaechtnissrede auf Eytelwein
merged_df.loc[merged_df["ID"] == "094927847", "SP"] = "XV"
merged_df.loc[merged_df["ID"] == "094927847", "EP"] = "XXXIV"
# 07-abh 1858: Ueber die Laengenbestimmung von Berlin und Koenigsberg ... : Nachtrag
merged_df.loc[merged_df["ID"] == "094968357", "SP"] = "82"
merged_df.loc[merged_df["ID"] == "094968357", "EP"] = "99"

# 07-abh 1832-2: W. v. Humboldt, Ueber die Kawi-Sprache ... : Erster Band --
# Katalog-Umfang "XXVIII, CCCCXXX, 312 S." (drei Zaehlungen), daraus wurde
# 1-CCCCXXX; Startseite laut Pruefung im Viewer VII.
merged_df.loc[merged_df["ID"] == "094897611", "SP"] = "VII"

# 08-verh 1841: Dirichlet, Untersuchungen ueber die Theorie der complexen
# Zahlen : Auszug -- Katalog "S. 190-164" (Endseite vor Startseite);
# Endseite laut Pruefung im Viewer 194.
merged_df.loc[merged_df["ID"] == "094974802", "EP"] = "194"

# 02-hist 1751: Le Monnier, Avertissement [d'une erreur qui se trouve dans les
# Memoires 1750. p. 254] -- Katalog "S. 1-356"; laut Pruefung im Viewer 356-356.
merged_df.loc[merged_df["ID"] == "094935289", "SP"] = "356"
merged_df.loc[merged_df["ID"] == "094935289", "EP"] = "356"

# 08-verh 1837: Carus, Auffindung des ersten Ei- oder Dotterblaeschens ... --
# Katalog "S. 1-2001" (2001 = Digitalisierungsjahr); laut Pruefung im Viewer 91-93.
merged_df.loc[merged_df["ID"] == "094931178", "SP"] = "91"
merged_df.loc[merged_df["ID"] == "094931178", "EP"] = "93"

# 10-sitz 1893-1: Diels, Ueber das physikalische System des Straton --
# Katalog "S. 101-1241"; laut Pruefung im Viewer 101-127.
merged_df.loc[merged_df["ID"] == "094915059", "SP"] = "101"
merged_df.loc[merged_df["ID"] == "094915059", "EP"] = "127"

# --- Versatz-Ausreisser (03_metadata_analyse.ipynb), laut Pruefung im Viewer ---

# 02-hist 1745: Euler, Sur le mouvement des Corps flexibles (S. 54-55) ist
# richtig; die beiden Nachbarn stehen im Katalog als "S. 1-53" bzw. "S. 1-56".
merged_df.loc[merged_df["ID"] == "09489860X", "SP"] = "53"   # Euler, Sections Coniques : Auszug
merged_df.loc[merged_df["ID"] == "09489860X", "EP"] = "53"
merged_df.loc[merged_df["ID"] == "094931526", "SP"] = "56"   # Bernoulli, Nouveau Probleme de Mecanique : Auszug
merged_df.loc[merged_df["ID"] == "094931526", "EP"] = "56"

# 08-verh 1836: Ehrenberg, Ueber fossile Infusionsthiere (zwei Eintraege) --
# Katalog "S. 50-54" bzw. "S. 55-56".
merged_df.loc[merged_df["ID"] == "095005994", "SP"] = "55"
merged_df.loc[merged_df["ID"] == "095005994", "EP"] = "55"
merged_df.loc[merged_df["ID"] == "095006001", "SP"] = "83"
merged_df.loc[merged_df["ID"] == "095006001", "EP"] = "86"

# 08-verh 1844: Hagen, Ueber die Form und Staerke der gewoelbten Bogen : Auszug
# -- Katalog "S. 487-500".
merged_df.loc[merged_df["ID"] == "094968594", "SP"] = "48"
merged_df.loc[merged_df["ID"] == "094968594", "EP"] = "48"

# 08-verh 1844: Rose, Ueber die Titansaeure : Zweiter Theil -- Katalog "S. 248-252".
merged_df.loc[merged_df["ID"] == "09500601X", "SP"] = "286"
merged_df.loc[merged_df["ID"] == "09500601X", "EP"] = "290"

# 08-verh 1848: Ehrenberg, Neue Beobachtungen ueber das ... formenreiche Leben
# -- Katalog "S. 325-345".
merged_df.loc[merged_df["ID"] == "09497392X", "SP"] = "349"
merged_df.loc[merged_df["ID"] == "09497392X", "EP"] = "361"

# 09-mon 1877: Virchow, Zur Craniologie Illyriens -- Katalog "S. 789-819";
# Startseite 769, Endseite 819 stimmt (naechster Artikel beginnt auf S. 820).
merged_df.loc[merged_df["ID"] == "094920761", "SP"] = "769"

seiten_manuell_ids = set(merged_df.loc[
    (merged_df[["SP", "EP"]].fillna("") != seiten_vorher.fillna("")).any(axis=1), "ID"])
print(f"Seiten von Hand gesetzt/korrigiert: {len(seiten_manuell_ids)} Zeilen")


# --- Links (Bildnummer pn=) einzeln korrigieren -------------------------------
# Seitenzahlen stimmen, aber die Bildnummer im Katalog-Link ist falsch. Die
# richtige Bildnummer ergibt sich aus dem Versatz Bild - Seite der Nachbarartikel.

def setze_bildnummer(ppn, pn):
    maske = merged_df["ID"] == ppn
    assert maske.sum() == 1, ppn
    merged_df.loc[maske, "UR"] = merged_df.loc[maske, "UR"].str.replace(
        r"pn=\d+", f"pn={pn:07d}", regex=True)


# 09-mon 1861: Braun, Ueber eine sonderbare Wirkung der ... Spaetfroeste (S. 691-700)
# -- Katalog "pn=000070" (Ziffer fehlt); Versatz im Band 12 -> Bild 703.
setze_bildnummer("094954925", 703)
# 10-sitz 1890-2: Harnack, Antrittsrede (S. 788-791) -- Katalog "pn=0000739";
# Versatz im Band -549 -> Bild 239.
setze_bildnummer("094933391", 239)
# 10-sitz 1894-2: Schulze, Ueber die Abwaertsbiegung des Schwanztheiles ...
# (S. 1133-1134) -- Katalog "pn=0000684" (Ziffern vertauscht); Versatz -485 -> Bild 648.
setze_bildnummer("09494783X", 648)


Seiten von Hand gesetzt/korrigiert: 56 Zeilen


In [472]:
# Fortsetzung-Duplikate: SP/EP korrigieren
#
# Bei Artikeln mit Fortsetzung gibt es oft zwei (selten drei) Zeilen mit
# identischem T1/A1/Fortsetzung, aber unterschiedlichem UR (=unterschiedliche
# Scan-Bildnummer "pn="). Beide/alle tragen fälschlich dieselben SP/EP wie der
# Hauptteil. Die Zeile mit der niedrigsten pn ist der Hauptteil (SP/EP bleiben)
# -- alle anderen Zeilen der Gruppe bekommen die aus "Fortsetzung" geparsten
# Seitenangaben als SP/EP (überschreibt die falsch übernommenen Werte).
# Zeilen mit Einzelkorrektur (seiten_manuell_ids) bleiben unangetastet.

def extrahiere_pn(ur):
    m = re.search(r"pn=(\d+)", ur) if pd.notna(ur) else None
    return int(m.group(1)) if m else None


def parse_fortsetzung_seiten(text):
    t = re.sub(r"^ebenda\.\s*", "", text.strip(), flags=re.IGNORECASE).strip()
    werte = re.findall(r"\d+|[IVXLCDM]+", t)
    if not werte:
        return None, None
    if "-" in t and len(werte) >= 2:
        return werte[0], werte[1]
    if re.search(r"S\.?\s*$", t):
        # "<Wert> S." = Seitenzahl einer separat gezählten Fortsetzung (z.B. Anhang)
        return "1", werte[0]
    # blosse Zahl ohne "S." = Einzelseite, auf der die Fortsetzung steht
    return werte[0], werte[0]


merged_df["pn_tmp"] = merged_df["UR"].apply(extrahiere_pn)

korrigiert = 0
for _, gruppe in merged_df[merged_df["Fortsetzung"].notna()].groupby(["T1", "A1", "Fortsetzung"]):
    if len(gruppe) < 2:
        continue
    gruppe_sortiert = gruppe.sort_values("pn_tmp")
    fortsetzung_text = gruppe_sortiert.iloc[0]["Fortsetzung"]
    sp_neu, ep_neu = parse_fortsetzung_seiten(fortsetzung_text)
    if sp_neu is None:
        continue
    for idx in gruppe_sortiert.index[1:]:
        if merged_df.loc[idx, "ID"] in seiten_manuell_ids:
            continue  # Einzelkorrektur aus "Seiten manuell nachtragen" hat Vorrang
        merged_df.loc[idx, "SP"] = sp_neu
        merged_df.loc[idx, "EP"] = ep_neu
        korrigiert += 1

merged_df = merged_df.drop(columns=["pn_tmp"])
print(f"SP/EP korrigiert bei {korrigiert} Fortsetzungs-Duplikaten")


SP/EP korrigiert bei 21 Fortsetzungs-Duplikaten


In [473]:
# Verdoppelte erste Ziffer in Seitenzahlen korrigieren
#
# Im Katalog ist bei rund 100 meist einseitigen Eintraegen die erste Ziffer der
# Seitenzahl verdoppelt ("S. 1104" statt "S. 104", "S. 556" statt "S. 56").
# Erkennbar am Versatz zwischen Bildnummer (pn) und Startseite: Er weicht vom
# vorigen UND folgenden Artikel im Band ab, waehrend diese beiden
# uebereinstimmen. Korrigiert wird nur, wenn die Seite ohne die verdoppelte
# Ziffer genau zum Versatz der Nachbarn passt (Toleranz KORR_TOLERANZ Seiten).
# Mehrere Durchlaeufe, weil zwei fehlerhafte Nachbarn sich gegenseitig
# verdecken koennen. Von Hand gesetzte Seiten bleiben unangetastet.
# Korrigierte Zeilen: "Seiten_Quelle" = "korrigiert".

KORR_TOLERANZ = 3


def ohne_doppelziffer(wert):
    s = str(wert)
    if len(s) >= 2 and s.isdigit() and s[0] == s[1]:
        return int(s[1:])
    return None


seiten_korrigiert_ids = set()
for durchlauf in range(3):
    t = merged_df.assign(
        SP_i=pd.to_numeric(merged_df["SP"], errors="coerce"),
        pn_i=pd.to_numeric(merged_df["UR"].str.extract(r"pn=(\d+)")[0], errors="coerce"),
    ).dropna(subset=["SP_i", "pn_i"]).sort_values(["SR", "BD", "pn_i"])
    t["versatz"] = t["pn_i"] - t["SP_i"]
    g = t.groupby(["SR", "BD"])["versatz"]
    t["vor"], t["nach"] = g.shift(1), g.shift(-1)
    kandidaten = t[((t["vor"] - t["nach"]).abs() <= KORR_TOLERANZ)
                   & ((t["versatz"] - t["vor"]).abs() > KORR_TOLERANZ)
                   & ~t["ID"].isin(seiten_manuell_ids)]

    korrigiert = 0
    for idx, row in kandidaten.iterrows():
        sp_neu = ohne_doppelziffer(row["SP"])
        if sp_neu is None or abs(sp_neu - (row["pn_i"] - row["vor"])) > KORR_TOLERANZ:
            continue
        ep_neu = ohne_doppelziffer(row["EP"]) if pd.notna(row["EP"]) else None
        merged_df.loc[idx, "SP"] = str(sp_neu)
        if ep_neu is not None and ep_neu >= sp_neu:
            merged_df.loc[idx, "EP"] = str(ep_neu)
        seiten_korrigiert_ids.add(row["ID"])
        korrigiert += 1
    if korrigiert == 0:
        break

print(f"Verdoppelte erste Ziffer korrigiert: {len(seiten_korrigiert_ids)} Zeilen")


# "S. 1NNN" = "S. 1-NNN" bei eigener Seitenzaehlung je Abhandlung
#
# In manchen Baenden (z.B. 07-abh 18991900) beginnt jede Abhandlung mit S. 1.
# Dort fehlt im Katalog gelegentlich der Bindestrich: "S. 1114" meint S. 1-114.
# Korrigiert wird nur eine einseitige Angabe "1NNN" (NNN >= 10), wenn ein
# Nachbarartikel (nach Bildnummer) mit S. 1 beginnt UND der Abstand zur
# naechsten Bildnummer zum Umfang NNN passt (NNN-3 bis NNN+30 Bilder, wegen Tafeln).

t = merged_df.assign(
    pn_i=pd.to_numeric(merged_df["UR"].str.extract(r"pn=(\d+)")[0], errors="coerce"),
).dropna(subset=["pn_i"]).sort_values(["SR", "BD", "pn_i"])
g = t.groupby(["SR", "BD"])
t["sp_vor"], t["sp_nach"] = g["SP"].shift(1), g["SP"].shift(-1)
t["pn_nach"] = g["pn_i"].shift(-1)
kandidaten = t[(t["SP"] == t["EP"]) & t["SP"].str.fullmatch(r"1\d{2,}", na=False)
               & ((t["sp_vor"] == "1") | (t["sp_nach"] == "1"))
               & ~t["ID"].isin(seiten_manuell_ids)]

bindestrich_korrigiert = 0
for idx, row in kandidaten.iterrows():
    n = int(row["SP"][1:])
    bilder = row["pn_nach"] - row["pn_i"]
    if n < 10 or (pd.notna(bilder) and not (n - 3 <= bilder <= n + 30)):
        continue
    merged_df.loc[idx, "SP"] = "1"
    merged_df.loc[idx, "EP"] = str(n)
    seiten_korrigiert_ids.add(row["ID"])
    bindestrich_korrigiert += 1

print(f"Fehlender Bindestrich ('S. 1NNN' -> 1-NNN) korrigiert: {bindestrich_korrigiert} Zeilen")


Verdoppelte erste Ziffer korrigiert: 95 Zeilen
Fehlender Bindestrich ('S. 1NNN' -> 1-NNN) korrigiert: 1 Zeilen


In [474]:
# Fehlende Seitenzahlen aus der Bildnummer schaetzen
#
# Bei rund 100 Katalogzeilen fehlen Start- und Endseite schon im GBV-Katalog
# (weder "Umfang" noch "Bibliogr. Zusammenhang", siehe 03_metadata_analyse.ipynb).
# Umkehrung der UR-Schaetzung fuer NEU-Zeilen (weiter unten): Startseite = pn minus Versatz (pn - SP) des
# Artikels im selben Band mit der naechstgelegenen Bildnummer. Endseite =
# Startseite des im Band folgenden Artikels minus 1 (mindestens die Startseite).
# Test in der Analyse: ca. 80 % exakt, ca. 90 % hoechstens 2 Seiten daneben.
# Geschaetzte Werte werden in "Seiten_Quelle" als "geschaetzt" markiert
# (Katalogwerte: "csv", von Hand gesetzte: "manuell", Ziffer korrigiert: "korrigiert"). ak-gesch bleibt ausgenommen (Einzelmonographien,
# Seiten werden dort weiter unten bewusst geleert).
#
# Laeuft VOR der Verweisaufloesung, damit Verweise auf diese Zeilen (z.B.
# "Auszug: 1853. MB S. 123") ein Ziel finden und keine NEU-Kopie entsteht.
# Ergibt die Schaetzung mehr als MAX_SPAN_GESCHAETZT Seiten (typisch: letzter
# Artikel vor einer separat paginierten Abteilung), bleibt die Endseite leer;
# 03_manifest_enrich.py begrenzt die Range dann ueber den naechsten Artikel.

#
# Katalogfehler bei einem Nachbarn (z.B. "33-303" statt "333-363") wuerden
# sich in die Schaetzung fortpflanzen. Deshalb werden Versatz-Ausreisser nicht
# als Nachbarn verwendet: Zeilen, deren Versatz von dem des vorigen UND des
# folgenden Artikels (nach Bildnummer) abweicht, waehrend diese beiden
# untereinander uebereinstimmen (Toleranz VERSATZ_TOLERANZ Seiten).

#
# Eigene Seitenzaehlung je Abhandlung (z.B. 07-abh 18991900, Gedaechtnisreden):
# Beginnen der vorige und der folgende Artikel (nach Bildnummer) mit S. 1,
# ist der Versatz zwischen Bild und Seite bedeutungslos. Dann gilt Startseite 1,
# die Endseite bleibt leer (der Bildabstand streut wegen nicht katalogisierter
# Tafeln um mehrere Seiten); 03_manifest_enrich.py begrenzt die Range ueber
# den naechsten Artikel.
#
# Vor einem Neubeginn der Zaehlung (FOLGENDER Artikel beginnt mit S. 1):
#   - Am Bandanfang (kein voriger Artikel): Vorspann mit eigener, meist
#     roemischer Zaehlung (z.B. Gedaechtnisreden). Keine Schaetzung, Seiten
#     bleiben leer (Nachtrag als Einzelkorrektur nach Pruefung im Viewer).
#   - Mitten im Band: Der Eintrag setzt die Zaehlung des VORIGEN Artikels fort
#     (bestaetigt an Encke 1844/1845/1858). Startseite ueber dessen Versatz,
#     Endseite bleibt leer, weil danach die Zaehlung neu beginnt.

MAX_SPAN_GESCHAETZT = 150
VERSATZ_TOLERANZ = 3

merged_df["SP_i"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df["pn_i"] = pd.to_numeric(merged_df["UR"].str.extract(r"pn=(\d+)")[0], errors="coerce")
merged_df["Seiten_Quelle"] = np.where(merged_df["SP"].notna(), "csv", None)
merged_df.loc[merged_df["ID"].isin(seiten_manuell_ids), "Seiten_Quelle"] = "manuell"
merged_df.loc[merged_df["ID"].isin(seiten_korrigiert_ids), "Seiten_Quelle"] = "korrigiert"

seiten_basis = merged_df[
    merged_df["SP_i"].notna() & merged_df["pn_i"].notna()
].copy()
seiten_basis["versatz"] = seiten_basis["pn_i"] - seiten_basis["SP_i"]


def versatz_ausreisser(gruppe):
    v = gruppe.sort_values("pn_i")["versatz"]
    vor, nach = v.shift(1), v.shift(-1)
    return ((vor - nach).abs() <= VERSATZ_TOLERANZ) & ((v - vor).abs() > VERSATZ_TOLERANZ)


ausreisser = seiten_basis.groupby(["SR", "BD"], group_keys=False).apply(versatz_ausreisser)
seiten_basis = seiten_basis[~ausreisser.reindex(seiten_basis.index, fill_value=False)]
print(f"Versatz-Ausreißer (nicht als Nachbarn verwendet): {int(ausreisser.sum())}")

ohne_seiten = merged_df[
    merged_df["SP"].isna() & merged_df["pn_i"].notna() & (merged_df["SR"] != "ak-gesch")
]

# 1. Startseiten
sp_geschaetzt = {}
eigene_paginierung = set()
vorspann = set()
vor_neubeginn = set()
for idx, row in ohne_seiten.iterrows():
    im_band = merged_df[(merged_df["SR"] == row["SR"]) & (merged_df["BD"] == row["BD"])
                        & merged_df["SP"].notna() & merged_df["pn_i"].notna()]
    vor = im_band[im_band["pn_i"] < row["pn_i"]].sort_values("pn_i").tail(1)
    nach = im_band[im_band["pn_i"] > row["pn_i"]].sort_values("pn_i").head(1)
    vor_sp1 = len(vor) > 0 and vor["SP"].iloc[0] == "1"
    nach_sp1 = len(nach) > 0 and nach["SP"].iloc[0] == "1"
    if vor_sp1 and nach_sp1:
        sp_geschaetzt[idx] = 1
        eigene_paginierung.add(idx)
        continue
    if nach_sp1 and len(vor) == 0:
        vorspann.add(idx)
        continue
    if nach_sp1 and pd.notna(vor["SP_i"].iloc[0]):
        sp_geschaetzt[idx] = max(1, int(row["pn_i"] - (vor["pn_i"].iloc[0] - vor["SP_i"].iloc[0])))
        vor_neubeginn.add(idx)
        continue
    if nach_sp1:
        vorspann.add(idx)  # voriger Artikel roemisch gezaehlt
        continue

    nachbarn = seiten_basis[(seiten_basis["SR"] == row["SR"]) & (seiten_basis["BD"] == row["BD"])]
    if nachbarn.empty:
        continue
    nachbar = nachbarn.loc[(nachbarn["pn_i"] - row["pn_i"]).abs().idxmin()]
    sp_geschaetzt[idx] = max(1, int(row["pn_i"] - nachbar["versatz"]))

# 2. Endseiten: naechster Artikel im Band (bekannte und geschaetzte Startseiten)
starts = merged_df.loc[merged_df["pn_i"].notna(), ["SR", "BD", "pn_i", "SP_i"]].copy()
starts.loc[list(sp_geschaetzt), "SP_i"] = pd.Series(sp_geschaetzt)
starts = starts.dropna(subset=["SP_i"])

for idx, sp in sp_geschaetzt.items():
    row = merged_df.loc[idx]
    folgende = starts[(starts["SR"] == row["SR"]) & (starts["BD"] == row["BD"]) & (starts["pn_i"] > row["pn_i"])]
    ep = max(int(folgende.loc[folgende["pn_i"].idxmin(), "SP_i"]) - 1, sp) if not folgende.empty else None
    if ep is not None and ep - sp > MAX_SPAN_GESCHAETZT:
        ep = None
    if idx in eigene_paginierung or idx in vor_neubeginn:
        ep = None
    merged_df.loc[idx, "SP"] = str(sp)
    merged_df.loc[idx, "EP"] = str(ep) if ep is not None else None
    merged_df.loc[idx, "Seiten_Quelle"] = "geschaetzt"

merged_df = merged_df.drop(columns=["SP_i", "pn_i"])
print(f"Seiten geschätzt für {len(sp_geschaetzt)} von {len(ohne_seiten)} Zeilen ohne Seitenangabe "
      f"(davon ohne Endseite: {sum(merged_df.loc[list(sp_geschaetzt), 'EP'].isna())}, "
      f"davon eigene Paginierung mit Startseite 1: {len(eigene_paginierung)})")
print(f"Startseite aus vorigem Artikel (vor Neubeginn der Zählung, Endseite offen): {len(vor_neubeginn)}")
print(f"Nicht geschätzt (Vorspann / eigene, vermutlich römische Zählung): {len(vorspann)}")

Versatz-Ausreißer (nicht als Nachbarn verwendet): 8
Seiten geschätzt für 96 von 96 Zeilen ohne Seitenangabe (davon ohne Endseite: 0, davon eigene Paginierung mit Startseite 1: 0)
Startseite aus vorigem Artikel (vor Neubeginn der Zählung, Endseite offen): 0
Nicht geschätzt (Vorspann / eigene, vermutlich römische Zählung): 0


In [475]:
# Unveraenderte Anmerkung sichern (fuer spaetere Rekonstruktion der Textbeziehungen)
merged_df["Anmerkung_original_backup"] = merged_df["Anmerkung"]


In [476]:
# Anmerkung: interne Verweise aufloesen (einzeln und mehrfach), unaufgeloeste als neue Zeilen anlegen
#
# Viele Anmerkungen (z.B. "Vgl. 1860. MB S. 1-3", "Auszug: 1837. MB S. 62-64;
# 95-103; 528-535", "Ebenda. S. 302-310") verweisen auf andere, bereits in
# merged_df enthaltene Zeilen. Ein Chunk kann dabei mehrere Seitenbereiche zu
# EINEM Jahr/Kuerzel nennen; jeder einzelne Bereich wird separat aufgeloest:
# existiert dafuer schon eine Zeile (eindeutig, gleicher A1, plausible Spanne
# -- MAX_SPAN_2 filtert Datenfehler wie ID 094931178 mit EP=2001 als Ziel
# aus), wird er aus der Anmerkung entfernt. Existiert noch keine, bleibt der
# Bereich in der Anmerkung stehen UND wird als neue Zeile angelegt (T1/A1 von
# der Quellzeile uebernommen, SR/BD/SP/EP aus dem Verweis). Chunks mit
# mehreren Jahres-/Kuerzel-Gruppen ohne " - "-Trennung (z.B. "...579-580.
# 1841. MB S. 101-102...") sind zu mehrdeutig und werden unveraendert gelassen.
#
# Neu angelegte Zeilen bekommen eine synthetische ID (Praefix "NEU-", keine
# echte GBV-ID) und keinen UR-Link, da der Zusammenhang zwischen gedruckter
# Seitenzahl und Scan-Bildnummer (pn=) nicht zuverlaessig rekonstruierbar ist.
# Herkunft_ID/Herkunft_Chunk verweisen auf die Quellzeile zur Kontrolle.
#
# Damit keine Kopien bestehender Zeilen entstehen (siehe 03_metadata_analyse.ipynb):
#   - Gibt es keinen eindeutigen Treffer mit gleichem Autor, zaehlt auch eine
#     Zeile mit exakt gleicher Fundstelle (Reihe, Band, Start- und Endseite)
#     als Ziel -- bei mehreren identischen die mit der niedrigsten Bildnummer.
#   - Passen mehrere Zeilen desselben Autors, gilt die mit der naechstgelegenen
#     Fundstelle (eindeutig, hoechstens 3 Seiten Abweichung).
#   - Abhandlungen (07-abh) erscheinen ein Jahr nach ihrem Jahrgang; Verweise
#     nennen oft das Erscheinungsjahr ("Vgl. 1851. Math. Abh. S. 1-60" meint
#     Band 1850). Findet sich im genannten Band kein Ziel, wird der Band des
#     Vorjahres geprueft.
#   - Verweist ein Bereich auf die Seiten der Quellzeile selbst, ist das ein
#     Selbstverweis: kein Ziel, keine neue Zeile.
#   - Pro Fundstelle (Reihe, Band, Start- und Endseite) wird nur EINE neue
#     Zeile angelegt, auch wenn mehrere Quellzeilen sie nennen (z.B. eine
#     Aufsatzreihe, deren Teile wechselseitig aufeinander verweisen).
#     Herkunft_ID listet dann alle Quellzeilen; neu_herkunft_index ordnet
#     jedem (Quell-ID, Teilverweis) die neue Zeile zu (fuer die Textbeziehungen).

MEM_REIHEN = ["02-hist", "03-nouv", "05-mem"]

ABBR_LOOKUP_2 = {
    "MB": (["09-mon", "08-verh"], None),
    "SB I.": (["10-sitz"], "1"), "SB I": (["10-sitz"], "1"), "SB l.": (["10-sitz"], "1"),
    "SB II.": (["10-sitz"], "2"), "SB II": (["10-sitz"], "2"),
    "Hist.": (MEM_REIHEN, None),
    "Phys. Abh.": (["07-abh"], None), "Math. Abh.": (["07-abh"], None),
    "Philol.-hist. Abh.": (["07-abh"], None), "Hist.-philol. Abh.": (["07-abh"], None),
    "Phys.-math. Abh.": (["07-abh"], None), "Philos.-hist. Abh.": (["07-abh"], None),
    "Hist.-philos. Abh.": (["07-abh"], None), "Philos. Abh.": (["07-abh"], None), "Abh.": (["07-abh"], None),
    # "Mém."/"Hist." meinen je nach Jahr verschiedene Reihen: bis 1769 die
    # Histoire (02-hist), 1770-1786 die Nouveaux Mémoires (03-nouv), danach die
    # Mémoires (05-mem, Doppeljahrgaenge wie "17881789"). Welche Reihe gemeint
    # ist, entscheidet der tatsaechlich vorhandene Band (siehe resolve_band).
    "Mém. Classe de math.": (MEM_REIHEN, None), "Mém. Classe de belles-lettres.": (MEM_REIHEN, None),
    "Mém. Classe de philos. spécul.": (MEM_REIHEN, None), "Mém. Classe de philos. expér.": (MEM_REIHEN, None),
    "Mém.": (MEM_REIHEN, None),
}
ABBR_ALTERNATION_2 = "|".join(re.escape(k) for k in sorted(ABBR_LOOKUP_2, key=len, reverse=True))
HEAD_RE = re.compile(
    rf"^(?P<label>Vgl\.?:?|Vergl\.?:?|vgl\.?:?|Auszug:?|Zusatz:?|Fortsetzung:?|Nachtrag:?)?\s*"
    rf"(?P<jahr>\d{{4}})\.\s*(?P<abbr>{ABBR_ALTERNATION_2})\s*S\.\s*(?P<tail>.+)$"
)
EMBEDDED_HEAD_RE = re.compile(rf"\d{{4}}\.\s*(?:{ABBR_ALTERNATION_2})\s*S\.")
EBENDA_HEAD_RE_2 = re.compile(r"^Ebenda\.?\s*S\.\s*(?P<tail>.+)$")
CHUNK_SPLIT_RE_2 = re.compile(r"\s-\s")
TOKEN_SPLIT_RE = re.compile(r"[;:]")
TOKEN_NUM_RE = re.compile(r"^(\d+)\s*-?\s*(\d+)?$")
MAX_SPAN_2 = 400

merged_df["SP_i"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df["EP_i"] = pd.to_numeric(merged_df["EP"], errors="coerce")


def surname_2(a1):
    return a1.split(",")[0].strip().lower() if pd.notna(a1) else ""


def naechste_fundstelle(kandidaten, p1, p2, toleranz=3):
    """Bei mehreren Treffern desselben Autors: die Zeile, deren Seiten der
    Fundstelle eindeutig am naechsten liegen (Abweichung <= toleranz Seiten)."""
    abstand = ((kandidaten["SP_i"] - p1).abs() + (kandidaten["EP_i"] - p2).abs()).sort_values()
    if len(abstand) >= 1 and abstand.iloc[0] <= toleranz and (len(abstand) == 1 or abstand.iloc[1] > abstand.iloc[0]):
        return kandidaten.loc[[abstand.index[0]]]
    return None


def find_target_2(sr_list, bd, p1, p2, own_id, own_author, nur_start=False):
    sub = merged_df[merged_df["SR"].isin(sr_list) & (merged_df["BD"] == bd)]
    sub = sub[(sub["SP_i"] <= p2) & (sub["EP_i"] >= p1)]
    sub = sub[(sub["EP_i"] - sub["SP_i"]) <= MAX_SPAN_2]
    sub = sub[sub["ID"] != own_id]
    gleicher_autor = sub[sub["A1"].apply(surname_2) == surname_2(own_author)]
    if len(gleicher_autor) == 1:
        return gleicher_autor
    if len(gleicher_autor) > 1 and nur_start:
        # Endseite des Verweises unbrauchbar: Zeile, die genau auf p1 beginnt
        beginnt = gleicher_autor[gleicher_autor["SP_i"] == p1]
        if len(beginnt) == 1:
            return beginnt
    if len(gleicher_autor) > 1:
        naechste = naechste_fundstelle(gleicher_autor, p1, p2)
        if naechste is not None:
            return naechste
    # Fallback: exakt gleiche Fundstelle (unabhaengig vom Autor)
    exakt = sub[(sub["SP_i"] == p1) & (sub["EP_i"] == p2)]
    if len(exakt) >= 1:
        pn = pd.to_numeric(exakt["UR"].str.extract(r"pn=(\d+)")[0], errors="coerce")
        return exakt.loc[[pn.fillna(float("inf")).idxmin()]]
    return gleicher_autor


def vorjahresband(sr_list, jahr, baende):
    """Band des Vorjahres fuer Verweise auf die Abhandlungen (07-abh), falls vorhanden."""
    if sr_list != ["07-abh"] or jahr is None:
        return None
    vorjahr = str(int(jahr) - 1)
    k = baende[(baende["SR"] == "07-abh") & baende["BD"].apply(band_passt, jahr=vorjahr)]
    return k.iloc[0]["BD"] if len(k) == 1 else None


def ist_selbstverweis(row, sr_list, bd, p1, p2):
    return (row["SR"] in sr_list and row["BD"] == bd
            and pd.notna(row["SP_i"]) and pd.notna(row["EP_i"])
            and row["SP_i"] <= p2 and row["EP_i"] >= p1)


def band_passt(band, jahr):
    """Band 'jahr' oder Doppeljahrgang wie '17881789', der 'jahr' umfasst."""
    if band == jahr:
        return True
    m = re.fullmatch(r"(\d{4})(\d{4})", str(band))
    return bool(m) and int(m.group(1)) <= int(jahr) <= int(m.group(2))


BAENDE_2 = merged_df[["SR", "BD"]].dropna().drop_duplicates()


def resolve_band_2(sr_list, jahr, teil):
    """(Reihe, Band) fuer einen Verweis 'Jahr. Kuerzel' -- nur wenn eindeutig."""
    if teil:
        bd = f"{jahr}-{teil}"
        kandidaten = BAENDE_2[BAENDE_2["SR"].isin(sr_list) & (BAENDE_2["BD"] == bd)]
    else:
        kandidaten = BAENDE_2[BAENDE_2["SR"].isin(sr_list) & BAENDE_2["BD"].apply(band_passt, jahr=jahr)]
    if len(kandidaten) == 1:
        return kandidaten.iloc[0]["SR"], kandidaten.iloc[0]["BD"]
    if len(sr_list) == 1 and kandidaten.empty:
        # wie bisher: eindeutige Reihe, Band (noch) nicht im Bestand
        return sr_list[0], (f"{jahr}-{teil}" if teil else jahr)
    return None


neue_zeilen = []
neu_nach_fundstelle = {}   # (SR, BD, SP, EP) -> neue Zeile (dict)
neu_herkunft_index = {}    # (Quell-ID, Teilverweis) -> ID der neuen Zeile
anmerkung_teilweise = []
selbstverweise = 0
for _, row in merged_df.iterrows():
    text = row["Anmerkung"]
    if pd.isna(text):
        anmerkung_teilweise.append(np.nan)
        continue

    out_chunks = []
    for chunk in CHUNK_SPLIT_RE_2.split(text.strip()):
        s = chunk.strip()
        sr_list = bd = tail = head_text = jahr = None

        m = EBENDA_HEAD_RE_2.match(s)
        if m:
            if pd.notna(row["SR"]) and pd.notna(row["BD"]):
                sr_list, bd, tail = [row["SR"]], row["BD"], m.group("tail")
                head_text = "Ebenda. S."
        else:
            m2 = HEAD_RE.match(s)
            if m2 and not EMBEDDED_HEAD_RE.search(m2.group("tail")):
                jahr, abbr = m2.group("jahr"), m2.group("abbr")
                kandidaten_sr, teil = ABBR_LOOKUP_2[abbr]
                aufgeloest = resolve_band_2(kandidaten_sr, jahr, teil)
                if aufgeloest is not None:
                    sr, bd = aufgeloest
                    sr_list, tail = [sr], m2.group("tail")
                    label = (m2.group("label") or "").strip()
                    head_text = (label + " " if label else "") + f"{jahr}. {abbr} S."

        if sr_list is None:
            out_chunks.append(chunk)
            continue

        kept_tokens, any_removed = [], False
        for tok in (t.strip() for t in TOKEN_SPLIT_RE.split(tail)):
            mnum = TOKEN_NUM_RE.match(tok.rstrip(" ."))
            if not mnum:
                kept_tokens.append(tok)
                continue
            p1 = int(mnum.group(1))
            p2 = int(mnum.group(2)) if mnum.group(2) else p1
            nur_start = p2 < p1
            if p2 < p1:
                # Endseite im Verweis fehlerhaft (z.B. "S. 179-18" statt 179-189):
                # nur die Startseite verwenden
                p2 = p1
            treffer = find_target_2(sr_list, bd, p1, p2, row["ID"], row["A1"], nur_start=nur_start)
            if len(treffer) != 1:
                bd_vorjahr = vorjahresband(sr_list, jahr, BAENDE_2)
                if bd_vorjahr is not None:
                    treffer_vj = find_target_2(sr_list, bd_vorjahr, p1, p2, row["ID"], row["A1"], nur_start=nur_start)
                    if len(treffer_vj) == 1:
                        treffer = treffer_vj
            if len(treffer) == 1:
                any_removed = True
            elif ist_selbstverweis(row, sr_list, bd, p1, p2):
                any_removed = True
                selbstverweise += 1
            else:
                kept_tokens.append(tok)
                fundstelle = (sr_list[0], bd, p1, p2)
                if fundstelle in neu_nach_fundstelle:
                    neue_zeile = neu_nach_fundstelle[fundstelle]
                    if row["ID"] not in neue_zeile["Herkunft_ID"].split(", "):
                        neue_zeile["Herkunft_ID"] += f", {row['ID']}"
                else:
                    neue_zeile = {
                        "ID": f"NEU-{row['ID']}-{len(neue_zeilen) + 1}",
                        "T1": row["T1"], "A1": row["A1"],
                        "SR": sr_list[0], "BD": bd, "SP": str(p1), "EP": str(p2),
                        "Herkunft_ID": row["ID"], "Herkunft_Chunk": tok,
                        "Seiten_Quelle": "verweis",
                    }
                    neue_zeilen.append(neue_zeile)
                    neu_nach_fundstelle[fundstelle] = neue_zeile
                neu_herkunft_index[(row["ID"], tok)] = neue_zeile["ID"]

        if any_removed and kept_tokens:
            out_chunks.append(f"{head_text} " + "; ".join(kept_tokens))
        elif not any_removed:
            out_chunks.append(chunk)

    anmerkung_teilweise.append(" - ".join(out_chunks) if out_chunks else np.nan)

merged_df["Anmerkung"] = anmerkung_teilweise
merged_df = merged_df.drop(columns=["SP_i", "EP_i"])

if neue_zeilen:
    merged_df = pd.concat([merged_df, pd.DataFrame(neue_zeilen)], ignore_index=True)

print(f"Neu angelegte Zeilen aus unaufgelösten Teilverweisen: {len(neue_zeilen)}")
print(f"Verworfene Selbstverweise: {selbstverweise}")
print(f"Davon von mehreren Quellzeilen genannt: "
      f"{sum(', ' in z['Herkunft_ID'] for z in neue_zeilen)}")


Neu angelegte Zeilen aus unaufgelösten Teilverweisen: 32
Verworfene Selbstverweise: 39
Davon von mehreren Quellzeilen genannt: 9


In [477]:
# Antwort: neue Zeilen fuer die Antworten auf Antrittsreden anlegen
#
# "Antwort" enthaelt Text wie "Antwort von BÖCKH: Ebenda. S. 217" -- eine
# Antwort auf die Antrittsrede der Quellzeile, im selben SR/BD (Ebenda), auf
# den genannten Seiten. Viele dieser Antworten sind aber schon als eigene
# Zeile katalogisiert (z.B. "Antwort auf die Antrittsreden von X, Y und Z"),
# oft weil ein/e Sekretär*in in einer Sitzung auf mehrere Antrittsreden
# gleichzeitig antwortete. Erst pruefen, ob eine solche Zeile schon existiert
# (SR/BD/Seiten-Overlap + Nachname), nur bei Nichtexistenz eine neue Zeile
# anlegen. Mehrere Quellzeilen mit identischem Antwort-Text (SR/BD/Name/
# Seiten) teilen sich EINE neue Zeile (wie die echten Vorbilder im Datensatz).
# Der volle Name der/des Antwortenden wird, wo moeglich, aus einer bereits
# bestehenden Zeile mit demselben Nachnamen uebernommen (sonst nur Nachname).
#
# Auf jeder Quellzeile verweist die neue Spalte "Antwort_ID" auf die
# (bestehende oder neu angelegte) Zielzeile.

ANTWORT_PAT = re.compile(
    r"^(Antwort von|Autwort von|Résponse de)\s+([^:.]+)[:.]\s*Ebenda\.?\s*S[.,]\s*(\d+)\s*-?\s*(\d+)?\s*$"
)
ANTWORT_ALIAS = {"ENCK": "ENCKE", "DUBOISREMOND": "DUBOISREYMOND"}


def antwort_norm_name(s):
    key = re.sub(r"[^A-ZÀ-Ü]", "", s.upper())
    return ANTWORT_ALIAS.get(key, key)


def antwort_surname_key(a1):
    return antwort_norm_name(a1.split(",")[0].strip()) if pd.notna(a1) else ""


merged_df["SP_i"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df["EP_i"] = pd.to_numeric(merged_df["EP"], errors="coerce")

antwort_zeilen = merged_df[merged_df["Antwort"].notna()]
name_map = {}
antwort_id = {}
unaufgeloeste_gruppen = {}

for idx, row in antwort_zeilen.iterrows():
    m = ANTWORT_PAT.match(row["Antwort"].strip())
    if not m:
        continue
    key_name = antwort_norm_name(m.group(2).strip())
    p1 = int(m.group(3))
    p2 = int(m.group(4)) if m.group(4) else p1
    sr, bd = row["SR"], row["BD"]

    sub = merged_df[(merged_df["SR"] == sr) & (merged_df["BD"] == bd)]
    sub = sub[(sub["SP_i"] <= p2) & (sub["EP_i"] >= p1)]
    sub = sub[sub["A1"].apply(antwort_surname_key) == key_name]

    if len(sub) == 1:
        antwort_id[idx] = sub.iloc[0]["ID"]
        name_map.setdefault(key_name, sub.iloc[0]["A1"])
    else:
        gruppen_key = (sr, bd, key_name, p1, p2)
        unaufgeloeste_gruppen.setdefault(gruppen_key, []).append(idx)

neue_zeilen = []
for n, ((sr, bd, key_name, p1, p2), idxs) in enumerate(unaufgeloeste_gruppen.items(), start=1):
    quellen = merged_df.loc[idxs]
    surnames = [a1.split(",")[0].strip() for a1 in quellen["A1"]]
    if len(surnames) == 1:
        t1 = f"Antwort auf die Antrittsrede von {surnames[0]}"
    else:
        t1 = f"Antwort auf die Antrittsreden von {', '.join(surnames[:-1])} und {surnames[-1]}"

    neu_id = f"NEU-ANTWORT-{n}"
    neue_zeilen.append({
        "ID": neu_id, "T1": t1, "A1": name_map.get(key_name, key_name.title()),
        "SR": sr, "BD": bd, "SP": str(p1), "EP": str(p2),
        "Herkunft_ID": ", ".join(quellen["ID"]),
        "Herkunft_Chunk": quellen.iloc[0]["Antwort"],
        "Seiten_Quelle": "verweis",
    })
    for idx in idxs:
        antwort_id[idx] = neu_id

merged_df["Antwort_ID"] = pd.Series(antwort_id)
merged_df = merged_df.drop(columns=["SP_i", "EP_i"])

if neue_zeilen:
    merged_df = pd.concat([merged_df, pd.DataFrame(neue_zeilen)], ignore_index=True)

print(f"Bereits bestehende Antwort-Zeilen verknuepft: {len(antwort_id) - len(neue_zeilen)}")
print(f"Neu angelegte Antwort-Zeilen: {len(neue_zeilen)}")


Bereits bestehende Antwort-Zeilen verknuepft: 60
Neu angelegte Antwort-Zeilen: 35


In [478]:
# Textbeziehungen: alle intertextuellen Beziehungen in einer Spalte buendeln
#
# Fasst alle bisher ueber SR/BD/Seiten aufgeloesten Beziehungen als ID-basierte
# "Typ: ID"-Paare zusammen (getrennt mit " | "), auf BEIDEN beteiligten Zeilen:
#   - Vgl/Auszug/Zusatz/Fortsetzung/Nachtrag/Verweis: aus Anmerkung_original_backup
#     neu ermittelt (die Zelle "Anmerkung: interne Verweise aufloesen" hat
#     diese IDs bereits berechnet, aber nicht persistiert -- daher hier eine
#     eigene, in sich abgeschlossene Neuberechnung anhand des Backups, ohne
#     die vorherige Zelle zu veraendern).
#   - Antwort/Antwort auf: aus der Spalte Antwort_ID.
#   - Fortsetzung/Fortsetzung von: aus den Fortsetzung-Duplikat-Gruppen (wie in
#     der Zelle "Fortsetzung-Duplikate", hier erneut anhand T1/A1/Fortsetzung/
#     UR ermittelt).
#
# Beziehungstyp ist aus Sicht der jeweiligen Zeile formuliert, z.B. bedeutet
# "Fortsetzung: <ID>" auf Zeile A "die Fortsetzung von A steht in <ID>", und
# "Fortsetzung von: <ID>" auf Zeile B "B ist die Fortsetzung von <ID>".

from collections import defaultdict

TB_ABBR_LOOKUP = {
    "MB": (["09-mon", "08-verh"], None),
    "SB I.": (["10-sitz"], "1"), "SB I": (["10-sitz"], "1"), "SB l.": (["10-sitz"], "1"),
    "SB II.": (["10-sitz"], "2"), "SB II": (["10-sitz"], "2"),
    "Hist.": (MEM_REIHEN, None),
    "Phys. Abh.": (["07-abh"], None), "Math. Abh.": (["07-abh"], None),
    "Philol.-hist. Abh.": (["07-abh"], None), "Hist.-philol. Abh.": (["07-abh"], None),
    "Phys.-math. Abh.": (["07-abh"], None), "Philos.-hist. Abh.": (["07-abh"], None),
    "Hist.-philos. Abh.": (["07-abh"], None), "Philos. Abh.": (["07-abh"], None), "Abh.": (["07-abh"], None),
    # "Mém."/"Hist." meinen je nach Jahr verschiedene Reihen: bis 1769 die
    # Histoire (02-hist), 1770-1786 die Nouveaux Mémoires (03-nouv), danach die
    # Mémoires (05-mem, Doppeljahrgaenge wie "17881789"). Welche Reihe gemeint
    # ist, entscheidet der tatsaechlich vorhandene Band (siehe resolve_band).
    "Mém. Classe de math.": (MEM_REIHEN, None), "Mém. Classe de belles-lettres.": (MEM_REIHEN, None),
    "Mém. Classe de philos. spécul.": (MEM_REIHEN, None), "Mém. Classe de philos. expér.": (MEM_REIHEN, None),
    "Mém.": (MEM_REIHEN, None),
}
TB_ABBR_ALT = "|".join(re.escape(k) for k in sorted(TB_ABBR_LOOKUP, key=len, reverse=True))
TB_HEAD_RE = re.compile(
    rf"^(?P<label>Vgl\.?:?|Vergl\.?:?|vgl\.?:?|Auszug:?|Zusatz:?|Fortsetzung:?|Nachtrag:?)?\s*"
    rf"(?P<jahr>\d{{4}})\.\s*(?P<abbr>{TB_ABBR_ALT})\s*S\.\s*(?P<tail>.+)$"
)
TB_EMBEDDED_HEAD_RE = re.compile(rf"\d{{4}}\.\s*(?:{TB_ABBR_ALT})\s*S\.")
TB_EBENDA_HEAD_RE = re.compile(r"^Ebenda\.?\s*S\.\s*(?P<tail>.+)$")
TB_CHUNK_SPLIT_RE = re.compile(r"\s-\s")
TB_TOKEN_SPLIT_RE = re.compile(r"[;:]")
TB_TOKEN_NUM_RE = re.compile(r"^(\d+)\s*-?\s*(\d+)?$")
TB_MAX_SPAN = 400


def tb_surname(a1):
    return a1.split(",")[0].strip().lower() if pd.notna(a1) else ""


def tb_klassifiziere(label):
    # "Vgl"/"Vergl" wird bewusst NICHT als eigener Typ gefuehrt, sondern mit
    # dem allgemeinen "Verweis" zusammengefasst -- die Inhalte sind zu
    # uneinheitlich (mal Fortsetzungen ohne Teil1/Teil2-Kennzeichnung, mal
    # Duplikate, mal einfach dieselbe Stelle im Band), um einen praeziseren
    # eigenen Namen zu rechtfertigen.
    l = (label or "").lower().rstrip(": .").strip()
    if l.startswith("auszug"):
        return "Auszug"
    if l.startswith("zusatz"):
        return "Zusatz"
    if l.startswith("fortsetzung"):
        return "Fortsetzung"
    if l.startswith("nachtrag"):
        return "Nachtrag"
    return "Verweis"


merged_df["SP_i"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df["EP_i"] = pd.to_numeric(merged_df["EP"], errors="coerce")
tb_nicht_neu = merged_df[~merged_df["ID"].str.startswith("NEU-")]


def tb_find_target(sr_list, bd, p1, p2, own_id, own_author, nur_start=False):
    # gleiche Logik wie find_target_2 (inkl. Fallback auf exakt gleiche Fundstelle)
    sub = tb_nicht_neu[tb_nicht_neu["SR"].isin(sr_list) & (tb_nicht_neu["BD"] == bd)]
    sub = sub[(sub["SP_i"] <= p2) & (sub["EP_i"] >= p1)]
    sub = sub[(sub["EP_i"] - sub["SP_i"]) <= TB_MAX_SPAN]
    sub = sub[sub["ID"] != own_id]
    gleicher_autor = sub[sub["A1"].apply(tb_surname) == tb_surname(own_author)]
    if len(gleicher_autor) == 1:
        return gleicher_autor
    if len(gleicher_autor) > 1 and nur_start:
        # Endseite des Verweises unbrauchbar: Zeile, die genau auf p1 beginnt
        beginnt = gleicher_autor[gleicher_autor["SP_i"] == p1]
        if len(beginnt) == 1:
            return beginnt
    if len(gleicher_autor) > 1:
        naechste = naechste_fundstelle(gleicher_autor, p1, p2)
        if naechste is not None:
            return naechste
    exakt = sub[(sub["SP_i"] == p1) & (sub["EP_i"] == p2)]
    if len(exakt) >= 1:
        pn = pd.to_numeric(exakt["UR"].str.extract(r"pn=(\d+)")[0], errors="coerce")
        return exakt.loc[[pn.fillna(float("inf")).idxmin()]]
    return gleicher_autor


TB_BAENDE = tb_nicht_neu[["SR", "BD"]].dropna().drop_duplicates()


def tb_resolve_band(sr_list, jahr, teil):
    # gleiche Logik wie resolve_band_2
    if teil:
        bd = f"{jahr}-{teil}"
        kandidaten = TB_BAENDE[TB_BAENDE["SR"].isin(sr_list) & (TB_BAENDE["BD"] == bd)]
    else:
        kandidaten = TB_BAENDE[TB_BAENDE["SR"].isin(sr_list) & TB_BAENDE["BD"].apply(band_passt, jahr=jahr)]
    if len(kandidaten) == 1:
        return kandidaten.iloc[0]["SR"], kandidaten.iloc[0]["BD"]
    if len(sr_list) == 1 and kandidaten.empty:
        return sr_list[0], (f"{jahr}-{teil}" if teil else jahr)
    return None


# (Quell-ID, Teilverweis) -> neue Zeile; aus der Zelle "interne Verweise aufloesen"
# (dort koennen mehrere Quellzeilen auf dieselbe neue Zeile zeigen)
tb_herkunft_index = neu_herkunft_index

beziehungen_liste = []
for _, row in merged_df.iterrows():
    text = row["Anmerkung_original_backup"]
    if pd.isna(text):
        continue
    for chunk in TB_CHUNK_SPLIT_RE.split(text.strip()):
        s = chunk.strip()
        sr_list = bd = tail = label = jahr = None
        ebenda = False

        m = TB_EBENDA_HEAD_RE.match(s)
        if m:
            if pd.notna(row["SR"]) and pd.notna(row["BD"]):
                sr_list, bd, tail = [row["SR"]], row["BD"], m.group("tail")
                ebenda = True
        else:
            m2 = TB_HEAD_RE.match(s)
            if m2 and not TB_EMBEDDED_HEAD_RE.search(m2.group("tail")):
                jahr, abbr = m2.group("jahr"), m2.group("abbr")
                kand_sr, teil = TB_ABBR_LOOKUP[abbr]
                aufgeloest = tb_resolve_band(kand_sr, jahr, teil)
                if aufgeloest is not None:
                    sr, bd = aufgeloest
                    sr_list, tail = [sr], m2.group("tail")
                    label = m2.group("label")

        if sr_list is None:
            continue
        typ = "Verweis" if ebenda else tb_klassifiziere(label)

        for tok in (t.strip() for t in TB_TOKEN_SPLIT_RE.split(tail)):
            mnum = TB_TOKEN_NUM_RE.match(tok.rstrip(" ."))
            if not mnum:
                continue
            p1 = int(mnum.group(1))
            p2 = int(mnum.group(2)) if mnum.group(2) else p1
            nur_start = p2 < p1
            if p2 < p1:
                # Endseite im Verweis fehlerhaft (z.B. "S. 179-18" statt 179-189):
                # nur die Startseite verwenden
                p2 = p1
            treffer = tb_find_target(sr_list, bd, p1, p2, row["ID"], row["A1"], nur_start=nur_start)
            if len(treffer) != 1:
                # wie in "interne Verweise aufloesen": Abhandlungen auch im Vorjahresband suchen
                bd_vorjahr = vorjahresband(sr_list, jahr, TB_BAENDE)
                if bd_vorjahr is not None:
                    treffer_vj = tb_find_target(sr_list, bd_vorjahr, p1, p2, row["ID"], row["A1"], nur_start=nur_start)
                    if len(treffer_vj) == 1:
                        treffer = treffer_vj
            if len(treffer) == 1:
                beziehungen_liste.append((row["ID"], typ, treffer.iloc[0]["ID"]))
            elif ist_selbstverweis(row, sr_list, bd, p1, p2):
                continue
            else:
                ziel = tb_herkunft_index.get((row["ID"], tok))
                if ziel:
                    beziehungen_liste.append((row["ID"], typ, ziel))

# Antwort_ID -> Beziehung
for _, r in merged_df[merged_df["Antwort_ID"].notna()].iterrows():
    beziehungen_liste.append((r["ID"], "Antwort", r["Antwort_ID"]))


# Fortsetzung-Duplikate -> Beziehung
def tb_extrahiere_pn(ur):
    m = re.search(r"pn=(\d+)", ur) if pd.notna(ur) else None
    return int(m.group(1)) if m else None


merged_df["pn_tmp"] = merged_df["UR"].apply(tb_extrahiere_pn)
for _, g in merged_df[merged_df["Fortsetzung"].notna()].groupby(["T1", "A1", "Fortsetzung"]):
    if len(g) < 2:
        continue
    g_sortiert = g.sort_values("pn_tmp")
    haupt_id = g_sortiert.iloc[0]["ID"]
    for _, r in g_sortiert.iloc[1:].iterrows():
        beziehungen_liste.append((haupt_id, "Fortsetzung", r["ID"]))

# Übersetzung/Original -> Beziehung
#
# "Übersetzung unter dem Titel: X. Jahr. Kuerzel S. Seiten" (bzw. "Deutsch
# unter dem Titel"/"In franzoesischer Uebersetzung unter dem Titel") auf
# Zeile A bedeutet: eine Uebersetzung von A steht unter dem Titel X an
# dieser Stelle -> Beziehung (A, "Uebersetzung", Ziel). "Original unter dem
# Titel" (bzw. "Uebersetzung von X") bedeutet umgekehrt: A IST die
# Uebersetzung, das Original steht an dieser Stelle -> Quelle/Ziel vertauscht,
# damit "Uebersetzung"/"Uebersetzung von" in beiden Faellen dieselbe
# Bedeutung behalten.

TB_UEB_CITATION_RE = re.compile(rf"(\d{{4}})(?:/\d{{2,4}})?\.\s*({TB_ABBR_ALT})\s*S\.\s*(\d+)(?:\s*-\s*(\d+))?")
TB_UEB_SWAP_LABELS = ("Original unter dem Titel", "Übersetzung von")

for _, row in merged_df[merged_df["Übersetzung/Original"].notna()].iterrows():
    text = row["Übersetzung/Original"]
    swap = text.startswith(TB_UEB_SWAP_LABELS)
    for m in TB_UEB_CITATION_RE.finditer(text):
        jahr, abbr, p1s, p2s = m.groups()
        p1 = int(p1s)
        p2 = int(p2s) if p2s else p1
        kand_sr, teil = TB_ABBR_LOOKUP[abbr]
        aufgeloest = tb_resolve_band(kand_sr, jahr, teil)
        if aufgeloest is None:
            continue
        sr, bd = aufgeloest
        treffer = tb_find_target([sr], bd, p1, p2, row["ID"], row["A1"])
        if len(treffer) == 1:
            ziel = treffer.iloc[0]["ID"]
            paar = (ziel, row["ID"]) if swap else (row["ID"], ziel)
            beziehungen_liste.append((paar[0], "Übersetzung", paar[1]))

# Die Rueckrichtung muss ebenfalls benennen, WAS der jeweilige Zieltitel ist
# (nicht nur "X von" als Verweis auf die Vorwaertsrichtung):
#   - Uebersetzung -> Original (Zielzeile IST das Original)
#   - Auszug -> Volltext (Zielzeile IST die vollstaendige Fassung)
#   - Fortsetzung/Nachtrag -> Hauptteil (Zielzeile IST der Hauptteil, zu dem
#     die aktuelle Zeile Fortsetzung bzw. nachtraeglicher Zusatz ist)
#   - Antwort -> Antrittsrede (Zielzeile IST die beantwortete Antrittsrede)
#   - Verweis bleibt in beide Richtungen "Verweis" (die Beziehung ist zu
#     uneinheitlich fuer eine gerichtete Bezeichnung)
TB_REVERSE = {
    "Auszug": "Volltext", "Zusatz": "Zusatz von",
    "Fortsetzung": "Hauptteil", "Nachtrag": "Hauptteil",
    "Verweis": "Verweis", "Antwort": "Antrittsrede", "Übersetzung": "Original",
}

tb_dict = defaultdict(list)
for a, typ, b in dict.fromkeys(beziehungen_liste):
    tb_dict[a].append(f"{typ}: {b}")
    tb_dict[b].append(f"{TB_REVERSE[typ]}: {a}")

textbeziehungen = {k: " | ".join(dict.fromkeys(v)) for k, v in tb_dict.items()}
merged_df["Textbeziehungen"] = merged_df["ID"].map(textbeziehungen)
merged_df = merged_df.drop(columns=["SP_i", "EP_i", "pn_tmp", "Anmerkung_original_backup"])

print(f"Zeilen mit Textbeziehungen: {merged_df['Textbeziehungen'].notna().sum()}")


Zeilen mit Textbeziehungen: 876


In [479]:
# UR fuer neu angelegte Zeilen (NEU-*) aus der Seitenzahl herleiten
#
# Der Link enthaelt "pn=<Bildnummer>" -- die Nummer des Scan-Bildes, die von
# der gedruckten Seitenzahl (SP) um einen Versatz abweicht (Titelblaetter,
# Inhaltsverzeichnisse etc.). Dieser Versatz ist nicht im ganzen Band
# konstant (mehrere Test-Ansaetze mit einem einzigen Band-weiten Versatz
# bzw. mit Interpolation zwischen den Nachbarn schnitten schlechter ab), aber
# lokal meist stabil. Deshalb: fuer jede NEU-*-Zeile den existierenden
# Artikel im selben SR/BD mit der zahlenmaessig naechstgelegenen SP suchen
# und dessen Versatz (pn - SP) uebernehmen. Das ist eine Schaetzung, kein
# garantiert korrekter Link -- daher zusaetzlich als "UR_geschaetzt"
# markiert, damit das im Zweifel nachvollzogen/geprueft werden kann.

merged_df["SP_i"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df["pn_i"] = merged_df["UR"].str.extract(r"pn=(\d+)")[0]

bekannt = merged_df[
    ~merged_df["ID"].str.startswith("NEU-") & merged_df["UR"].notna() & merged_df["pn_i"].notna()
].copy()
bekannt["pn_i"] = bekannt["pn_i"].astype(int)

ur_geschaetzt = {}
for idx, row in merged_df[merged_df["ID"].str.startswith("NEU-")].iterrows():
    if pd.isna(row["SR"]) or pd.isna(row["BD"]) or pd.isna(row["SP_i"]):
        continue
    kandidaten = bekannt[(bekannt["SR"] == row["SR"]) & (bekannt["BD"] == row["BD"])]
    if kandidaten.empty:
        continue
    nachbar = kandidaten.loc[(kandidaten["SP_i"] - row["SP_i"]).abs().idxmin()]

    versatz = nachbar["pn_i"] - nachbar["SP_i"]
    pn_neu = max(1, round(row["SP_i"] + versatz))
    pn_string_alt = re.search(r"pn=(\d+)", nachbar["UR"]).group(1)
    pn_string_neu = str(int(pn_neu)).zfill(len(pn_string_alt))

    merged_df.loc[idx, "UR"] = re.sub(r"pn=\d+", f"pn={pn_string_neu}", nachbar["UR"])
    ur_geschaetzt[row["ID"]] = "ja"

merged_df["UR_geschaetzt"] = merged_df["ID"].map(ur_geschaetzt)
merged_df = merged_df.drop(columns=["SP_i", "pn_i"])

print(f"UR geschaetzt fuer {len(ur_geschaetzt)} von "
      f"{merged_df['ID'].str.startswith('NEU-').sum()} neu angelegten Zeilen")


UR geschaetzt fuer 67 von 67 neu angelegten Zeilen


In [480]:
merged_df = merged_df.drop(columns=["Anmerkung", "Fortsetzung", "Antwort", "Herkunft_ID", "Herkunft_Chunk", "Antwort_ID", "UR_geschaetzt", "Übersetzung/Original"])

In [481]:
# Titel normalisieren: Anfuehrungszeichen am Anfang/Ende entfernen, ersten
# Buchstaben grossschreiben.
#
# Wirkt sich damit auch auf die Textbeziehungen-Anzeige aus, da diese im
# Enrich-Schritt (03_manifest_enrich.py, beziehungen_html()) den Titel der
# jeweiligen Zielzeile hinter dem Beziehungstyp einsetzt -- ein normalisierter
# Titel liefert dort automatisch ein grossgeschriebenes erstes Wort ohne
# unpassende Anfuehrungszeichen.

QUOTE_CHARS = "\"'\u201e\u201c\u2018\u2019\u201d`\u00b4"

merged_df["T1"] = merged_df["T1"].str.strip(QUOTE_CHARS).str.strip()
merged_df["T1"] = merged_df["T1"].apply(
    lambda s: s[0].upper() + s[1:] if isinstance(s, str) and s else s
)


In [482]:
# AK-Geschichte kuratieren
merged_df = merged_df[~merged_df['UR'].str.contains('digitalequel', na=False)]

ak_gesch_mask = merged_df['SR'] == 'ak-gesch'

# Seitenzahlen und Band sind bei ak-gesch nicht sinnvoll (kein Sammelband mit
# durchgezaehlten Abhandlungen wie bei den anderen Schriftenreihen)
merged_df.loc[ak_gesch_mask, ['SP', 'EP', 'Seiten_Quelle']] = None

# Titel/Autor fuer die beiden unbetitelten Bartholmess-Baende nachtragen
merged_df.loc[ak_gesch_mask & (merged_df['T1'] == 'Teil 1'), ['T1', 'A1']] = [
    "Histoire philosophique de l'Académie de Prusse depuis Leibniz jusqu'a Schelling, particul. sous Frédéric-le-Grand [Teil 1]",
    "Bartholmèss, Christian",
]
merged_df.loc[ak_gesch_mask & (merged_df['T1'] == 'Teil 2'), ['T1', 'A1']] = [
    "Histoire philosophique de l'Académie de Prusse depuis Leibniz jusqu'a Schelling, particul. sous Frédéric-le-Grand [Teil 2]",
    "Bartholmèss, Christian",
]

# Erscheinungsjahr fuer den Bildband nachtragen
merged_df.loc[ak_gesch_mask & merged_df['T1'].str.contains('Bildnisse berühmter', na=False), 'PY'] = "1950"

# Eckige Klammern aus dem Erscheinungsjahr entfernen (z.B. "[1900]" -> "1900")
merged_df.loc[ak_gesch_mask, 'PY'] = merged_df.loc[ak_gesch_mask, 'PY'].str.replace(r'[\[\]]', '', regex=True)

In [483]:
merged_df["SP_sort"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df = merged_df.sort_values(by=["SR", "BD", "SP_sort"])
merged_df = merged_df.drop(columns=["SP_sort"])

# Sprechende Spaltennamen fuer den Export (nur hier, damit die Zellen
# oben (SP/EP/T1/A1/SR/BD/UR) unveraendert weiterfunktionieren)
merged_df = merged_df.rename(columns={
    "T1": "Titel",
    "A1": "Autor",
    "UR": "Link",
    "SR": "Schriftenreihe",
    "BD": "Band",
    "SP": "Startseite",
    "EP": "Endseite",
    "PY": "Jahr",
})


## Zweite Kuration

Bereinigt die fertig umbenannte Tabelle weiter, bevor sie gespeichert wird:

1. **Jahre ergänzen:** fehlende Werte in `Jahr` auffüllen; die neue Spalte `Jahr_Quelle` dokumentiert die Herkunft
2. **Autoren normalisieren:** `Autor` wird überschrieben (Korrekturliste, reine Nachnamen auflösen, Schreibvarianten vereinheitlichen)

In [484]:
import unicodedata

# Stand vor der zweiten Kuration (fuer die Kontrollzellen)
merged_df_vor_kuration = merged_df.copy()

#### 1. Jahre ergänzen

Fehlende Jahre werden in dieser Reihenfolge ergänzt:

1. häufigstes bekanntes Jahr innerhalb desselben Bandes (Reihe + Band) → `band_modus`
2. Jahreszahl am Anfang der Bandbezeichnung (z. B. `1882-1`, `1856`), aber nur, wenn sie im bekannten Zeitraum der Reihe liegt → `band_nummer`

In [485]:
def ergaenze_jahre(df):
    jahr = pd.to_numeric(df["Jahr"], errors="coerce").astype("Int64")

    quelle = jahr.notna().map({True: "csv", False: None})

    basis = df.assign(jahr=jahr).dropna(subset=["jahr"])
    band_modus = basis.groupby(["Schriftenreihe", "Band"])["jahr"].agg(lambda s: s.mode().iloc[0])
    reihen_zeitraum = basis.groupby("Schriftenreihe")["jahr"].agg(["min", "max"])

    for idx in df.index[jahr.isna()]:
        reihe, band = df.at[idx, "Schriftenreihe"], df.at[idx, "Band"]
        if (reihe, band) in band_modus.index:
            jahr.at[idx] = band_modus[(reihe, band)]
            quelle.at[idx] = "band_modus"
            continue
        m = re.match(r"^(1[6-9]\d\d)", str(band))
        if m and reihe in reihen_zeitraum.index:
            j = int(m.group(1))
            lo, hi = reihen_zeitraum.loc[reihe, "min"], reihen_zeitraum.loc[reihe, "max"]
            if lo <= j <= hi:
                jahr.at[idx] = j
                quelle.at[idx] = "band_nummer"

    df["Jahr"] = jahr
    df["Jahr_Quelle"] = quelle
    return df


merged_df = ergaenze_jahre(merged_df)
print(f"Jahr fehlt noch bei {merged_df['Jahr'].isna().sum()} Abhandlungen")
merged_df["Jahr_Quelle"].value_counts(dropna=False)

Jahr fehlt noch bei 0 Abhandlungen


Jahr_Quelle
csv           5885
band_modus    2384
Name: count, dtype: int64

In [486]:
# Abhandlungen, bei denen das Jahr weiterhin fehlt (nach Reihe/Band)
merged_df[merged_df["Jahr"].isna()].groupby(["Schriftenreihe", "Band"], dropna=False).size()

Series([], dtype: int64)

#### 2. Autoren normalisieren

- Manuelle Korrekturen für Namen, die nicht dem Schema „Nachname, Vorname“ folgen (`AUTOR_KORREKTUREN`)
- Reine Nachnamen (z. B. „Böckh“) werden zum vollen Namen aufgelöst, wenn das eindeutig möglich ist
- Schreibvarianten, die sich nur in Groß-/Kleinschreibung, Akzenten oder Satzzeichen unterscheiden (z. B. „Du Bois-Reymond“ / „du Bois-Reymond“), werden zur häufigsten Schreibweise zusammengeführt
- „Ohne Angabe Des Verfassers“ bleibt unverändert stehen, leere Autorfelder bleiben leer

In [487]:
ANONYM = "Ohne Angabe Des Verfassers"

# Manuelle Korrekturen fuer Autorennamen, die nicht dem Schema
# "Nachname, Vorname" folgen. Schluessel = Wert in der CSV.
AUTOR_KORREKTUREN = {
    "Friedrich Ii König von Preussen": "Friedrich II., Preußen, König",
    "Friedrich": "Friedrich II., Preußen, König",
    "Wilhelm Adolf Prinz von Braunschweig": "Wilhelm Adolf, Braunschweig-Wolfenbüttel, Prinz",
}


def vergleichs_key(name):
    """Interner Schluessel, um Schreibvarianten zu erkennen
    (ohne Akzente, Kleinschreibung, nur Buchstaben/Ziffern)."""
    s = unicodedata.normalize("NFKD", name)
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    return re.sub(r"[^a-z0-9]+", "-", s).strip("-")


def normalisiere_autoren(df):
    roh = df["Autor"].str.strip()
    voll = sorted({a for a in roh.dropna() if "," in a})
    nachname_zu_voll = {}
    for name in voll:
        nachname_zu_voll.setdefault(name.split(",")[0].strip(), set()).add(name)

    def normalisiere(name):
        if pd.isna(name) or name == ANONYM:
            return name
        if name in AUTOR_KORREKTUREN:
            return AUTOR_KORREKTUREN[name]
        if "," not in name:
            # Nur Nachname (z.B. 'Böckh'): eindeutig aufloesbar?
            kandidaten = nachname_zu_voll.get(name, set())
            if len(kandidaten) == 1:
                return next(iter(kandidaten))
        return name

    autor = roh.map(normalisiere)

    # Schreibvarianten mit gleichem Schluessel zusammenfuehren; es gilt die
    # haeufigste Schreibweise.
    key = autor.map(lambda a: vergleichs_key(a) if isinstance(a, str) else None)
    kanonisch = (pd.DataFrame({"key": key, "autor": autor}).dropna()
                   .groupby("key")["autor"].agg(lambda s: s.value_counts().index[0]))
    df["Autor"] = key.map(kanonisch).where(key.notna(), autor)
    return df


merged_df = normalisiere_autoren(merged_df)

geaendert = merged_df_vor_kuration["Autor"].fillna("") != merged_df["Autor"].fillna("")
print(f"Autor geändert in {geaendert.sum()} Zeilen")
print(f"Anonym: {(merged_df['Autor'] == ANONYM).sum()}, leer: {merged_df['Autor'].isna().sum()}, "
      f"nur Nachname (ohne Komma): {(merged_df['Autor'].notna() & ~merged_df['Autor'].str.contains(',', na=False) & (merged_df['Autor'] != ANONYM)).sum()}")

Autor geändert in 53 Zeilen
Anonym: 19, leer: 2, nur Nachname (ohne Komma): 7


In [488]:
# Alle Aenderungen im Ueberblick (alt -> neu, Anzahl Zeilen)
(pd.DataFrame({"alt": merged_df_vor_kuration.loc[geaendert, "Autor"], "neu": merged_df.loc[geaendert, "Autor"]})
   .value_counts().rename("Zeilen").reset_index())

,alt,neu,Zeilen
0,Friedrich Ii König von Preussen,"Friedrich II., Preußen, König",18
1,"Du Bois-Reymond, Emil","Du Bois-reymond, Emil",6
2,Böckh,"Böckh, August",6
3,Ehrenberg,"Ehrenberg, Christian Gottfried",6
4,Trendelenburg,"Trendelenburg, Friedrich Adolf",4
5,Encke,"Encke, Johann Franz",4
6,Kummer,"Kummer, Ernst Eduard",2
7,Wilhelm Adolf Prinz von Braunschweig,"Wilhelm Adolf, Braunschweig-Wolfenbüttel, Prinz",1
8,"Spiess, N.","Spiess, N",1
9,"Imhoof-blumer, Friedrich","Imhoof-Blumer, Friedrich",1


In [489]:
# Verbleibende Autoren ohne Komma (nicht eindeutig aufloesbar) -- Kandidaten fuer AUTOR_KORREKTUREN
ohne_komma = merged_df["Autor"].notna() & ~merged_df["Autor"].str.contains(",", na=False) & (merged_df["Autor"] != ANONYM)
merged_df.loc[ohne_komma, "Autor"].value_counts()

Autor
Krüger        2
Batigne       1
J. H. B.      1
Petriciolo    1
Afdejew       1
Waldkyer      1
Name: count, dtype: int64

### Speichern

In [490]:
merged_df.to_csv("../../data/metadata/curated/merged_df.csv", index=False, encoding="utf-8-sig")
print(f"{len(merged_df)} Zeilen gespeichert. Spalten: {list(merged_df.columns)}")

8269 Zeilen gespeichert. Spalten: ['Startseite', 'Endseite', 'Abbilder', 'ID', 'Titel', 'Autor', 'Jahr', 'Link', 'Schriftenreihe', 'Band', 'Seiten_Quelle', 'Textbeziehungen', 'Jahr_Quelle']
